# Hydra baseline training notebook

This notebook launches the current **Hydra v1 baseline BC training** path the repo already supports on **Kaggle**.

This version is intentionally Kaggle-only:

- Kaggle dataset autodetect under `/kaggle/input`
- native Hydra build/run inside `/kaggle/working`
- preflight first, then training
- inline TensorBoard + JSONL dashboard


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import pathlib
import shlex
import shutil
import subprocess
import sys
import textwrap
import threading
import time
import traceback
import uuid
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from queue import Empty, Queue
from typing import Any, Callable

START_DIR = pathlib.Path.cwd().resolve()
KAGGLE_WORKING_ROOT = pathlib.Path('/kaggle/working')
IS_KAGGLE = KAGGLE_WORKING_ROOT.exists()
REPO_ROOT = (KAGGLE_WORKING_ROOT / 'hydra') if IS_KAGGLE else START_DIR
RUNS_DIR = REPO_ROOT / 'notebooks' / 'runs'
NOTEBOOK_STATE_ROOT = (KAGGLE_WORKING_ROOT / 'hydra-notebook-state') if IS_KAGGLE else (START_DIR / '.hydra-notebook-state')
PERSISTENT_LOG_ROOT = NOTEBOOK_STATE_ROOT / 'persistent-logs'
DEBUG_BUNDLE_ROOT = NOTEBOOK_STATE_ROOT / 'debug-bundles'
SESSION_ID = f"{time.strftime('%Y%m%d-%H%M%S')}-{uuid.uuid4().hex[:8]}"
SESSION_METADATA_PATH = PERSISTENT_LOG_ROOT / 'session-metadata.jsonl'
CELL_EVENT_LOG_PATH = PERSISTENT_LOG_ROOT / 'cell-events.jsonl'
WIDGET_ACTION_LOG_PATH = PERSISTENT_LOG_ROOT / 'widget-actions.jsonl'
TRAINING_STREAM_LOG_PATH = PERSISTENT_LOG_ROOT / 'training-stream.log'
TRAINING_STREAM_JSONL_PATH = PERSISTENT_LOG_ROOT / 'training-stream.jsonl'
PREFLIGHT_STREAM_LOG_PATH = PERSISTENT_LOG_ROOT / 'preflight-stream.log'
PREFLIGHT_STREAM_JSONL_PATH = PERSISTENT_LOG_ROOT / 'preflight-stream.jsonl'
PROCESS_STATE_PATH = NOTEBOOK_STATE_ROOT / 'process-state.json'
TRAINING_ARTIFACT_BUNDLE_PATH = DEBUG_BUNDLE_ROOT / 'hydra-kaggle-training-artifacts.zip'
STDOUT_LOG_PATH = PERSISTENT_LOG_ROOT / 'notebook-stdout.log'
STDERR_LOG_PATH = PERSISTENT_LOG_ROOT / 'notebook-stderr.log'
LATEST_DEBUG_BUNDLE_PATH = DEBUG_BUNDLE_ROOT / 'hydra-kaggle-latest-debug-bundle.zip'
LOG_FILE_LOCK = threading.Lock()
AUTO_REFRESH_THREADS: dict[str, threading.Thread] = {}
AUTO_REFRESH_STOP_EVENTS: dict[str, threading.Event] = {}
AUTO_REFRESH_INTERVALS: dict[str, int] = {}
CELL_HOOK_STATE: dict[str, Any] = {
    'counter': 0,
    'active': None,
    'hooks_installed': False,
}

# Change these before you run anything heavy.
DATASET_PATH = pathlib.Path('/home/nikketryhard/Downloads/dataset')
KAGGLE_INPUT_ROOT = pathlib.Path('/kaggle/input')
OFFLINE_BUNDLE_DATASET_NAME = 'offline-bundle-bin'
OFFLINE_BUNDLE_FALLBACK_DATASET_NAMES = ['offline-bundle']
OUTPUT_DIR = REPO_ROOT / 'output' / 'notebook-baseline'
CONFIG_PATH = RUNS_DIR / 'baseline_bc_rtx6000.yaml'
DEVICE = 'cuda:0'
ACTIVE_TRAINING_PROCESS: subprocess.Popen[str] | None = None
ACTIVE_TRAINING_QUEUE: Queue | None = None
ACTIVE_TRAINING_THREADS: list[threading.Thread] = []
ACTIVE_PREFLIGHT_PROCESS: subprocess.Popen[str] | None = None
ACTIVE_PREFLIGHT_QUEUE: Queue | None = None
ACTIVE_PREFLIGHT_THREADS: list[threading.Thread] = []

# Baseline BC defaults. Preflight will tune the microbatch side, not these top-level knobs.
NUM_EPOCHS = 1
BATCH_SIZE = 8192
TRAIN_FRACTION = 0.9
AUGMENT = True
SEED = 42
BUFFER_GAMES = 8192
BUFFER_SAMPLES = 524288
ARCHIVE_QUEUE_BOUND = 128
NUM_THREADS: int | None = 46
TRAIN_MICROBATCH_SIZE: int | None = 320
VALIDATION_MICROBATCH_SIZE: int | None = 6144
MAX_SKIP_LOGS_PER_SOURCE = 32
LOG_EVERY_N_STEPS = 25
VALIDATE_EVERY_N_STEPS = 200
CHECKPOINT_EVERY_N_STEPS = 200
MAX_VALIDATION_SAMPLES = 8192
TENSORBOARD = True

# Push preflight hard enough to actually search the box.
PREFLIGHT_CANDIDATES = [4096, 3072, 2560, 2048, 1536, 1024, 768, 512, 384, 256, 192, 128, 96, 64]

MONITOR_INTERVAL_SEC = 2.0

ACTIVE_DATASET_HOST_PATH = DATASET_PATH

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def json_safe(value: Any) -> Any:
    if isinstance(value, pathlib.Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_safe(inner) for key, inner in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return repr(value)

def ensure_persistent_state_dirs() -> None:
    for path in [NOTEBOOK_STATE_ROOT, PERSISTENT_LOG_ROOT, DEBUG_BUNDLE_ROOT]:
        path.mkdir(parents=True, exist_ok=True)

def append_text_log(path: pathlib.Path, text: str) -> None:
    ensure_persistent_state_dirs()
    with LOG_FILE_LOCK:
        with path.open('a', encoding='utf-8') as handle:
            handle.write(text)

def append_jsonl_log(path: pathlib.Path, payload: dict[str, Any]) -> None:
    record = {
        'timestamp_utc': utc_now_iso(),
        'session_id': SESSION_ID,
        **json_safe(payload),
    }
    append_text_log(path, json.dumps(record, sort_keys=True) + '\n')

class TeeStream:
    def __init__(self, original: Any, log_path: pathlib.Path):
        self.original = original
        self.log_path = log_path
        self._lock = threading.Lock()

    def write(self, data: Any) -> int:
        text = data if isinstance(data, str) else str(data)
        if not text:
            return 0
        with self._lock:
            self.original.write(text)
            self.original.flush()
            append_text_log(self.log_path, text)
        return len(text)

    def flush(self) -> None:
        self.original.flush()

    def writelines(self, lines: list[str]) -> None:
        for line in lines:
            self.write(line)

    def isatty(self) -> bool:
        return bool(getattr(self.original, 'isatty', lambda: False)())

    def __getattr__(self, name: str) -> Any:
        return getattr(self.original, name)

def install_stream_tee(stream_name: str, log_path: pathlib.Path) -> None:
    current = getattr(sys, stream_name)
    original = current.original if isinstance(current, TeeStream) else current
    setattr(sys, stream_name, TeeStream(original, log_path))

def log_cell_event(event: str, **payload: Any) -> None:
    append_jsonl_log(CELL_EVENT_LOG_PATH, {'event': event, **payload})

def log_widget_action(event: str, action: str, **payload: Any) -> None:
    append_jsonl_log(WIDGET_ACTION_LOG_PATH, {'event': event, 'action': action, **payload})

def append_training_stream_output(prefix: str, line: str) -> None:
    stamped = f"[{utc_now_iso()}] [{SESSION_ID}] [{prefix}] {line}\n"
    append_text_log(TRAINING_STREAM_LOG_PATH, stamped)
    append_jsonl_log(TRAINING_STREAM_JSONL_PATH, {'event': 'training_stream', 'stream': prefix, 'line': line})

def append_preflight_stream_output(prefix: str, line: str) -> None:
    stamped = f"[{utc_now_iso()}] [{SESSION_ID}] [{prefix}] {line}\n"
    append_text_log(PREFLIGHT_STREAM_LOG_PATH, stamped)
    append_jsonl_log(PREFLIGHT_STREAM_JSONL_PATH, {'event': 'preflight_stream', 'stream': prefix, 'line': line})

def persistent_log_status_snapshot() -> dict[str, dict[str, Any]]:
    ensure_persistent_state_dirs()
    tracked_paths = {
        'session_metadata': SESSION_METADATA_PATH,
        'cell_events': CELL_EVENT_LOG_PATH,
        'widget_actions': WIDGET_ACTION_LOG_PATH,
        'training_stream_text': TRAINING_STREAM_LOG_PATH,
        'training_stream_jsonl': TRAINING_STREAM_JSONL_PATH,
        'preflight_stream_text': PREFLIGHT_STREAM_LOG_PATH,
        'preflight_stream_jsonl': PREFLIGHT_STREAM_JSONL_PATH,
        'stdout_tee': STDOUT_LOG_PATH,
        'stderr_tee': STDERR_LOG_PATH,
        'latest_debug_bundle': LATEST_DEBUG_BUNDLE_PATH,
    }
    snapshot: dict[str, dict[str, Any]] = {}
    for name, path in tracked_paths.items():
        snapshot[name] = {
            'path': str(path),
            'exists': path.exists(),
            'size_bytes': path.stat().st_size if path.exists() and path.is_file() else 0,
        }
    return snapshot

def load_process_state() -> dict[str, Any]:
    if not PROCESS_STATE_PATH.exists():
        return {}
    try:
        return json.loads(PROCESS_STATE_PATH.read_text())
    except Exception:
        return {}

def write_process_state(payload: dict[str, Any]) -> None:
    ensure_persistent_state_dirs()
    PROCESS_STATE_PATH.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True) + '\n')

def pid_is_alive(pid: int | None) -> bool:
    if pid is None or pid <= 0:
        return False
    try:
        os.kill(pid, 0)
    except OSError:
        return False
    return True

def process_state_snapshot(kind: str) -> dict[str, Any]:
    state = load_process_state()
    entry = state.get(kind)
    return entry if isinstance(entry, dict) else {}

def persist_process_state(kind: str, proc: subprocess.Popen[str], cmd: list[str], stream_log_path: pathlib.Path, stream_jsonl_path: pathlib.Path) -> None:
    state = load_process_state()
    state[kind] = {
        'pid': proc.pid,
        'running': proc.poll() is None,
        'returncode': None if proc.poll() is None else proc.returncode,
        'command': cmd,
        'config_path': str(CONFIG_PATH),
        'output_dir': str(OUTPUT_DIR),
        'stream_log_path': str(stream_log_path),
        'stream_jsonl_path': str(stream_jsonl_path),
        'started_at_utc': utc_now_iso(),
        'session_id': SESSION_ID,
    }
    write_process_state(state)

def finalize_process_state(kind: str, proc: subprocess.Popen[str] | None = None, stop_signal: str | None = None) -> None:
    state = load_process_state()
    entry = state.get(kind)
    if not isinstance(entry, dict):
        entry = {}
    if proc is not None:
        entry['pid'] = proc.pid
        entry['running'] = proc.poll() is None
        entry['returncode'] = None if proc.poll() is None else proc.returncode
    else:
        entry['running'] = pid_is_alive(entry.get('pid'))
        if not entry['running'] and entry.get('returncode') is None:
            entry['returncode'] = 'unknown_after_recovery'
    entry['updated_at_utc'] = utc_now_iso()
    if stop_signal is not None:
        entry['stop_signal'] = stop_signal
        entry['stop_requested_at_utc'] = utc_now_iso()
    state[kind] = entry
    write_process_state(state)

def recover_process_status(kind: str, proc: subprocess.Popen[str] | None) -> dict[str, Any]:
    if proc is not None:
        return {
            'pid': proc.pid,
            'running': proc.poll() is None,
            'returncode': None if proc.poll() is None else proc.returncode,
            'source': 'live_handle',
            'state': process_state_snapshot(kind),
        }
    entry = process_state_snapshot(kind)
    pid = entry.get('pid')
    running = pid_is_alive(pid)
    return {
        'pid': pid,
        'running': running,
        'returncode': None if running else entry.get('returncode'),
        'source': 'persisted_state' if entry else 'none',
        'state': entry,
    }

def stop_process_by_pid(pid: int | None, force_after_seconds: int = 10) -> str:
    if pid is None or pid <= 0:
        return 'no_pid'
    try:
        os.kill(pid, 15)
    except OSError:
        return 'not_running'
    deadline = time.time() + force_after_seconds
    while time.time() < deadline:
        if not pid_is_alive(pid):
            return 'terminated'
        time.sleep(0.2)
    try:
        os.kill(pid, 9)
    except OSError:
        return 'terminated'
    return 'killed'

def stop_auto_refresh(kind: str) -> None:
    stop_event = AUTO_REFRESH_STOP_EVENTS.get(kind)
    if stop_event is not None:
        stop_event.set()

def start_auto_refresh(kind: str, refresh_fn: Callable[[], None], is_running_fn: Callable[[], bool], interval_seconds: int) -> None:
    stop_auto_refresh(kind)
    stop_event = threading.Event()
    AUTO_REFRESH_STOP_EVENTS[kind] = stop_event
    AUTO_REFRESH_INTERVALS[kind] = int(interval_seconds)

    def runner() -> None:
        while not stop_event.wait(max(1, int(interval_seconds))):
            if not is_running_fn():
                break
            try:
                refresh_fn()
            except Exception as err:
                append_jsonl_log(WIDGET_ACTION_LOG_PATH, {'event': 'auto_refresh_error', 'action': kind, 'error': repr(err)})
                break
        stop_event.set()

    thread = threading.Thread(target=runner, daemon=True)
    AUTO_REFRESH_THREADS[kind] = thread
    thread.start()

def wrap_widget_callback(action: str, callback: Callable[..., Any]) -> Callable[..., Any]:
    def wrapped(event: Any = None) -> Any:
        event_details = {'event_payload': json_safe(event)}
        log_widget_action('widget_action_start', action, **event_details)
        try:
            result = callback(event)
        except BaseException as err:
            log_widget_action(
                'widget_action_error',
                action,
                **event_details,
                error=repr(err),
                traceback=''.join(traceback.format_exception(err)),
            )
            raise
        log_widget_action('widget_action_end', action, **event_details)
        return result
    return wrapped

def _cell_source_payload(raw_cell: str) -> dict[str, Any]:
    return {
        'cell_source': raw_cell,
        'cell_sha256': hashlib.sha256(raw_cell.encode('utf-8')).hexdigest() if raw_cell else None,
    }

def notebook_pre_run_cell(info: Any) -> None:
    raw_cell = getattr(info, 'raw_cell', '') or ''
    CELL_HOOK_STATE['counter'] += 1
    cell_payload = {
        'cell_index': CELL_HOOK_STATE['counter'],
        **_cell_source_payload(raw_cell),
        'started_at_utc': utc_now_iso(),
        'started_monotonic': time.perf_counter(),
    }
    CELL_HOOK_STATE['active'] = cell_payload
    log_cell_event('cell_start', **{key: value for key, value in cell_payload.items() if key != 'started_monotonic'})

def notebook_post_run_cell(result: Any) -> None:
    active = CELL_HOOK_STATE.get('active') or {}
    raw_cell = getattr(getattr(result, 'info', None), 'raw_cell', active.get('cell_source', '')) or ''
    err = getattr(result, 'error_in_exec', None) or getattr(result, 'error_before_exec', None)
    duration_ms = None
    if active.get('started_monotonic') is not None:
        duration_ms = round((time.perf_counter() - float(active['started_monotonic'])) * 1000.0, 2)
    payload = {
        'cell_index': active.get('cell_index'),
        **_cell_source_payload(raw_cell),
        'started_at_utc': active.get('started_at_utc'),
        'finished_at_utc': utc_now_iso(),
        'duration_ms': duration_ms,
        'execution_count': getattr(result, 'execution_count', None),
        'status': 'error' if err is not None else 'ok',
        'exception': repr(err) if err is not None else None,
        'traceback': ''.join(traceback.format_exception(err)) if err is not None else None,
    }
    log_cell_event('cell_end', **payload)
    CELL_HOOK_STATE['active'] = None

def install_ipython_cell_logging() -> None:
    if CELL_HOOK_STATE.get('hooks_installed'):
        return
    try:
        from IPython import get_ipython
    except Exception:
        return
    ip = get_ipython()
    if ip is None or getattr(ip, 'events', None) is None:
        return
    ip.events.register('pre_run_cell', notebook_pre_run_cell)
    ip.events.register('post_run_cell', notebook_post_run_cell)
    CELL_HOOK_STATE['hooks_installed'] = True

def install_persistent_notebook_logging() -> None:
    ensure_persistent_state_dirs()
    install_stream_tee('stdout', STDOUT_LOG_PATH)
    install_stream_tee('stderr', STDERR_LOG_PATH)
    install_ipython_cell_logging()

install_persistent_notebook_logging()
append_jsonl_log(
    SESSION_METADATA_PATH,
    {
        'event': 'session_start',
        'start_dir': str(START_DIR),
        'repo_root': str(REPO_ROOT),
        'is_kaggle': IS_KAGGLE,
        'persistent_log_root': str(PERSISTENT_LOG_ROOT),
        'debug_bundle_root': str(DEBUG_BUNDLE_ROOT),
    },
)

print('start dir  =', START_DIR)
print('repo      =', REPO_ROOT)
print('kaggle?   =', IS_KAGGLE)
print('dataset   =', DATASET_PATH)
print('kaggle    =', KAGGLE_INPUT_ROOT)
print('output    =', OUTPUT_DIR)
print('config    =', CONFIG_PATH)
print('state root=', NOTEBOOK_STATE_ROOT)
print('logs      =', PERSISTENT_LOG_ROOT)


## Repo bootstrap
This notebook uses a bundled offline Hydra repo from Kaggle input, builds the native `train` binary if needed, and runs there.

In [ ]:
def run_checked(cmd: list[str], cwd: pathlib.Path | None = None) -> subprocess.CompletedProcess:
    print('$', ' '.join(shlex.quote(part) for part in cmd))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr, file=sys.stderr)
        raise RuntimeError(
            f"command failed with exit code {result.returncode}\nSTDOUT:\n{result.stdout or '<empty>'}\nSTDERR:\n{result.stderr or '<empty>'}"
        )
    return result

def bundle_manifest_path(repo_root: pathlib.Path | None = None) -> pathlib.Path:
    root = repo_root or REPO_ROOT
    return root / 'bundle-manifest.json'

def compat_artifact_root(repo_root: pathlib.Path | None = None) -> pathlib.Path:
    root = repo_root or REPO_ROOT
    return root / 'dist' / 'kaggle-compat'

def compat_artifact_bin_path(repo_root: pathlib.Path | None = None) -> pathlib.Path:
    return compat_artifact_root(repo_root) / 'bin' / 'train'

def compat_artifact_lib_dir(repo_root: pathlib.Path | None = None) -> pathlib.Path:
    return compat_artifact_root(repo_root) / 'lib'

def runtime_manifest_path(repo_root: pathlib.Path | None = None) -> pathlib.Path:
    return compat_artifact_root(repo_root) / 'runtime-manifest.json'

def compat_metadata_path(filename: str, repo_root: pathlib.Path | None = None) -> pathlib.Path:
    return compat_artifact_root(repo_root) / filename

def read_json_file(path: pathlib.Path) -> dict[str, Any] | None:
    if not path.exists() or not path.is_file():
        return None
    return json.loads(path.read_text())

def canonical_manifest_text(manifest: dict[str, Any] | None) -> str | None:
    if manifest is None:
        return None
    return json.dumps(manifest, sort_keys=True)

def load_bundle_manifest(repo_root: pathlib.Path | None = None) -> dict[str, Any] | None:
    return read_json_file(bundle_manifest_path(repo_root))

def load_runtime_manifest(repo_root: pathlib.Path | None = None) -> dict[str, Any] | None:
    return read_json_file(runtime_manifest_path(repo_root))

def manifest_entries_by_path(manifest: dict[str, Any] | None) -> dict[str, dict[str, Any]]:
    if manifest is None:
        return {}
    entries = {}
    for entry in manifest.get('files', []):
        if isinstance(entry, dict) and entry.get('path'):
            entries[str(entry['path'])] = entry
    return entries

def validate_manifest_file(root: pathlib.Path, relative_path: str, expected_size: int | None, label: str) -> list[str]:
    target = root / relative_path
    if not target.exists():
        return [f'{label} missing required path: {target}']
    if expected_size is not None and target.is_file() and target.stat().st_size != int(expected_size):
        return [
            f'{label} size mismatch for {target}: expected {expected_size} bytes, found {target.stat().st_size}'
        ]
    return []

def validate_bundle_manifest_contract(repo_root: pathlib.Path, manifest: dict[str, Any] | None) -> list[str]:
    if manifest is None:
        return [f'bundle manifest missing under {repo_root}']
    files_by_path = manifest_entries_by_path(manifest)
    required_paths = [str(path) for path in manifest.get('required_files', [])]
    if not required_paths:
        required_paths = sorted(files_by_path)
    errors: list[str] = []
    for relative_path in required_paths:
        entry = files_by_path.get(relative_path, {})
        errors.extend(validate_manifest_file(repo_root, relative_path, entry.get('size_bytes'), 'bundle manifest'))
    return errors

def validate_runtime_manifest_contract(repo_root: pathlib.Path, manifest: dict[str, Any] | None = None) -> list[str]:
    runtime_manifest = manifest or load_runtime_manifest(repo_root)
    if runtime_manifest is None:
        return [f'runtime manifest missing under {repo_root}']
    errors: list[str] = []
    binary = runtime_manifest.get('binary', {})
    binary_path = str(binary.get('path', 'bin/train'))
    errors.extend(validate_manifest_file(compat_artifact_root(repo_root), binary_path, binary.get('size_bytes'), 'runtime manifest'))
    lib_dir = compat_artifact_root(repo_root) / str(runtime_manifest.get('lib_dir', 'lib'))
    for lib_entry in runtime_manifest.get('required_runtime_libraries', []):
        if not isinstance(lib_entry, dict):
            continue
        relative_path = str(lib_entry.get('path', ''))
        if not relative_path:
            continue
        errors.extend(validate_manifest_file(compat_artifact_root(repo_root), relative_path, lib_entry.get('size_bytes'), 'runtime manifest'))
    for metadata_entry in runtime_manifest.get('metadata_files', []):
        if not isinstance(metadata_entry, dict):
            continue
        relative_path = str(metadata_entry.get('path', ''))
        if not relative_path:
            continue
        errors.extend(validate_manifest_file(compat_artifact_root(repo_root), relative_path, metadata_entry.get('size_bytes'), 'runtime manifest'))
    for sentinel in runtime_manifest.get('required_runtime_sentinels', []):
        sentinel_path = lib_dir / str(sentinel)
        if not sentinel_path.exists():
            errors.append(f'runtime manifest sentinel missing: {sentinel_path}')
    return errors

def repo_checkout_mode(repo_root: pathlib.Path | None = None) -> str:
    root = repo_root or REPO_ROOT
    if bundle_manifest_path(root).exists():
        return 'runtime_only_bundle'
    if (root / 'Cargo.toml').exists():
        return 'full_repo'
    return 'unknown'

def validate_full_repo_checkout(repo_root: pathlib.Path) -> list[str]:
    errors: list[str] = []
    required_paths = [
        repo_root / 'Cargo.toml',
        repo_root / 'notebooks' / 'hydra_baseline_training.ipynb',
        repo_root / 'crates' / 'hydra-train',
    ]
    for path in required_paths:
        if not path.exists():
            errors.append(f'full repo checkout missing required path: {path}')
    return errors

def validate_runtime_bundle_checkout(
    repo_root: pathlib.Path,
    expected_bundle_manifest: dict[str, Any] | None = None,
    expected_runtime_manifest: dict[str, Any] | None = None,
) -> list[str]:
    errors: list[str] = []
    local_bundle_manifest = load_bundle_manifest(repo_root)
    local_runtime_manifest = load_runtime_manifest(repo_root)
    if expected_bundle_manifest is not None and canonical_manifest_text(local_bundle_manifest) != canonical_manifest_text(expected_bundle_manifest):
        errors.append(f'bundle manifest mismatch for {repo_root}')
    if expected_runtime_manifest is not None and canonical_manifest_text(local_runtime_manifest) != canonical_manifest_text(expected_runtime_manifest):
        errors.append(f'runtime manifest mismatch for {repo_root}')
    errors.extend(validate_bundle_manifest_contract(repo_root, local_bundle_manifest or expected_bundle_manifest))
    errors.extend(validate_runtime_manifest_contract(repo_root, local_runtime_manifest or expected_runtime_manifest))
    return errors

def load_payload_bundle_manifest(payload_path: pathlib.Path) -> tuple[dict[str, Any], str]:
    with zipfile.ZipFile(payload_path) as zf:
        manifest_name = next((name for name in sorted(zf.namelist()) if name.rstrip('/') == 'hydra/bundle-manifest.json'), None)
        if manifest_name is None:
            manifest_name = next((name for name in sorted(zf.namelist()) if name.endswith('/bundle-manifest.json') or name == 'bundle-manifest.json'), None)
        if manifest_name is None:
            raise RuntimeError(f'offline bundle payload {payload_path} is missing bundle-manifest.json')
        return json.loads(zf.read(manifest_name).decode('utf-8')), manifest_name

def load_payload_runtime_manifest(payload_path: pathlib.Path) -> tuple[dict[str, Any], str]:
    with zipfile.ZipFile(payload_path) as zf:
        manifest_name = next((name for name in sorted(zf.namelist()) if name.rstrip('/') == 'hydra/dist/kaggle-compat/runtime-manifest.json'), None)
        if manifest_name is None:
            manifest_name = next((name for name in sorted(zf.namelist()) if name.endswith('/dist/kaggle-compat/runtime-manifest.json') or name == 'dist/kaggle-compat/runtime-manifest.json'), None)
        if manifest_name is None:
            raise RuntimeError(f'offline bundle payload {payload_path} is missing runtime-manifest.json')
        return json.loads(zf.read(manifest_name).decode('utf-8')), manifest_name

def refresh_repo_globals(repo_root: pathlib.Path | None = None) -> None:
    global REPO_ROOT, RUNS_DIR, OUTPUT_DIR, CONFIG_PATH
    if repo_root is not None:
        REPO_ROOT = repo_root
    RUNS_DIR = REPO_ROOT / 'notebooks' / 'runs'
    OUTPUT_DIR = REPO_ROOT / 'output' / 'notebook-baseline'
    CONFIG_PATH = RUNS_DIR / 'baseline_bc_rtx6000.yaml'
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def detect_kaggle_repo_bundle_payload() -> pathlib.Path | None:
    if not KAGGLE_INPUT_ROOT.exists():
        return None
    roots = [KAGGLE_INPUT_ROOT / OFFLINE_BUNDLE_DATASET_NAME]
    roots.extend(KAGGLE_INPUT_ROOT / name for name in OFFLINE_BUNDLE_FALLBACK_DATASET_NAMES)
    roots.extend([child for child in sorted(KAGGLE_INPUT_ROOT.iterdir()) if child.is_dir() and child not in roots])
    for preferred_root in roots:
        if not preferred_root.exists():
            continue
        direct_payload = preferred_root / 'hydra-kaggle-offline-bundle.bin'
        if direct_payload.exists():
            return direct_payload
        for candidate in sorted(preferred_root.rglob('hydra-kaggle-offline-bundle.bin')):
            return candidate
    return None

def unpack_offline_repo_payload(payload_path: pathlib.Path) -> pathlib.Path:
    extract_root = KAGGLE_WORKING_ROOT / 'offline_bundle_extract'
    payload_manifest, _ = load_payload_bundle_manifest(payload_path)
    payload_runtime_manifest, _ = load_payload_runtime_manifest(payload_path)
    repo_root = extract_root / str(payload_manifest.get('root_dir', 'hydra'))
    if repo_root.exists():
        extracted_bundle_manifest = load_bundle_manifest(repo_root)
        extracted_runtime_manifest = load_runtime_manifest(repo_root)
        extracted_errors = validate_runtime_bundle_checkout(repo_root, payload_manifest, payload_runtime_manifest)
        if canonical_manifest_text(extracted_bundle_manifest) == canonical_manifest_text(payload_manifest) and not extracted_errors:
            return repo_root
        print('refreshing extracted offline bundle because existing extract is stale:')
        for error in extracted_errors or ['bundle manifest mismatch']:
            print('-', error)
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(payload_path) as zf:
        for info in zf.infolist():
            name = info.filename.rstrip('/')
            if not name:
                continue
            target = extract_root / name
            is_dir = info.is_dir() or info.filename.endswith('/')
            if is_dir:
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.exists() and target.is_dir():
                shutil.rmtree(target)
            with zf.open(info) as source, open(target, 'wb') as dst:
                shutil.copyfileobj(source, dst)
    extracted_bundle_manifest = load_bundle_manifest(repo_root)
    extracted_runtime_manifest = load_runtime_manifest(repo_root)
    extracted_errors = validate_runtime_bundle_checkout(repo_root, payload_manifest, payload_runtime_manifest)
    if canonical_manifest_text(extracted_bundle_manifest) != canonical_manifest_text(payload_manifest) or extracted_errors:
        joined = '\n'.join(f'- {error}' for error in extracted_errors) or '- manifest mismatch after extraction'
        raise RuntimeError(f'offline bundle extraction failed manifest validation under {repo_root}\n{joined}')
    return repo_root

def detect_kaggle_repo_bundle() -> pathlib.Path | None:
    if not KAGGLE_INPUT_ROOT.exists():
        return None
    payload = detect_kaggle_repo_bundle_payload()
    if payload is not None:
        print('using opaque offline bundle payload from', payload)
        return unpack_offline_repo_payload(payload)
    preferred_roots = [KAGGLE_INPUT_ROOT / OFFLINE_BUNDLE_DATASET_NAME]
    preferred_roots.extend(KAGGLE_INPUT_ROOT / name for name in OFFLINE_BUNDLE_FALLBACK_DATASET_NAMES)
    search_roots = [root for root in preferred_roots if root.exists()]
    search_roots.extend([child for child in sorted(KAGGLE_INPUT_ROOT.iterdir()) if child.is_dir() and child not in preferred_roots])
    for root in search_roots:
        for candidate in [root / 'hydra', root]:
            if bundle_manifest_path(candidate).exists() and runtime_manifest_path(candidate).exists():
                if not validate_runtime_bundle_checkout(candidate):
                    return candidate
            if (candidate / 'Cargo.toml').exists():
                return candidate
        for bundle_manifest in sorted(root.rglob('bundle-manifest.json')):
            candidate = bundle_manifest.parent
            if runtime_manifest_path(candidate).exists() and not validate_runtime_bundle_checkout(candidate):
                return candidate
        for cargo_toml in sorted(root.rglob('Cargo.toml')):
            try:
                text = cargo_toml.read_text()
            except Exception:
                continue
            if 'members = ["crates/hydra-core", "crates/hydra-engine", "crates/hydra-train"]' in text:
                return cargo_toml.parent
    return None

def ensure_repo_checkout() -> pathlib.Path:
    global REPO_ROOT
    current_mode = repo_checkout_mode(REPO_ROOT)
    if not IS_KAGGLE:
        if current_mode == 'runtime_only_bundle':
            errors = validate_runtime_bundle_checkout(REPO_ROOT)
            if errors:
                raise RuntimeError('Local runtime-only bundle checkout is invalid:\n' + '\n'.join(f'- {error}' for error in errors))
            refresh_repo_globals(REPO_ROOT)
            return REPO_ROOT
        if current_mode == 'full_repo':
            errors = validate_full_repo_checkout(REPO_ROOT)
            if errors:
                raise RuntimeError('Local Hydra repo checkout is invalid:\n' + '\n'.join(f'- {error}' for error in errors))
            refresh_repo_globals(REPO_ROOT)
            return REPO_ROOT
        raise RuntimeError(f'Hydra repo or runtime bundle not found at {REPO_ROOT}. Open the notebook from the repo root or set REPO_ROOT manually.')
    bundled_repo = detect_kaggle_repo_bundle()
    if bundled_repo is None:
        raise RuntimeError('No bundled offline Hydra runtime/repo was found under /kaggle/input. Attach the offline bundle dataset that contains hydra-kaggle-offline-bundle.bin, or mount a hydra/ checkout or runtime bundle tree.')
    source_mode = repo_checkout_mode(bundled_repo)
    if source_mode == 'unknown':
        raise RuntimeError(f'Bundled Hydra checkout at {bundled_repo} is neither a full repo nor a runtime-only bundle.')
    source_bundle_manifest = load_bundle_manifest(bundled_repo) if source_mode == 'runtime_only_bundle' else None
    source_runtime_manifest = load_runtime_manifest(bundled_repo) if source_mode == 'runtime_only_bundle' else None
    if REPO_ROOT.exists():
        reuse_errors: list[str] = []
        if source_mode == 'runtime_only_bundle':
            reuse_errors = validate_runtime_bundle_checkout(REPO_ROOT, source_bundle_manifest, source_runtime_manifest)
            if not reuse_errors:
                refresh_repo_globals(REPO_ROOT)
                return REPO_ROOT
        else:
            full_repo_errors = validate_full_repo_checkout(REPO_ROOT)
            if not full_repo_errors:
                refresh_repo_globals(REPO_ROOT)
                return REPO_ROOT
            reuse_errors = full_repo_errors
        print('refreshing repo checkout because:')
        for error in reuse_errors:
            print('-', error)
        shutil.rmtree(REPO_ROOT)
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if bundled_repo.resolve() != REPO_ROOT.resolve():
        shutil.copytree(bundled_repo, REPO_ROOT)
    refresh_repo_globals(REPO_ROOT)
    if source_mode == 'runtime_only_bundle':
        post_copy_errors = validate_runtime_bundle_checkout(REPO_ROOT, source_bundle_manifest, source_runtime_manifest)
    else:
        post_copy_errors = validate_full_repo_checkout(REPO_ROOT)
    if post_copy_errors:
        raise RuntimeError('Hydra checkout failed validation after refresh:\n' + '\n'.join(f'- {error}' for error in post_copy_errors))
    return REPO_ROOT

def make_binary_executable(path: pathlib.Path) -> pathlib.Path:
    mode = path.stat().st_mode
    path.chmod(mode | 0o111)
    return path

def ensure_native_train_binary() -> pathlib.Path:
    repo_root = ensure_repo_checkout()
    compat_binary = compat_artifact_bin_path(repo_root)
    bundled_binary = repo_root / 'train'
    built_binary = repo_root / 'target' / 'release' / 'train'
    if compat_binary.exists() and compat_binary.is_file():
        return make_binary_executable(compat_binary)
    if built_binary.exists() and built_binary.is_file():
        return make_binary_executable(built_binary)
    cargo_path = shutil.which('cargo')
    if cargo_path is not None and repo_checkout_mode(repo_root) == 'full_repo':
        env = os.environ.copy()
        wrapper = repo_root / 'scripts' / 'rustc-wrapper.sh'
        if wrapper.exists():
            env['RUSTC_WRAPPER'] = str(wrapper)
        print('building native Hydra train binary with Kaggle cargo...')
        result = subprocess.run(
            [cargo_path, 'build', '--release', '--bin', 'train', '-p', 'hydra-train'],
            cwd=repo_root,
            env=env,
            text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f'cargo build failed with exit code {result.returncode}')
        if not built_binary.exists():
            raise RuntimeError(f'expected native train binary at {built_binary} after build')
        return make_binary_executable(built_binary)
    if bundled_binary.exists() and bundled_binary.is_file():
        return make_binary_executable(bundled_binary)
    raise RuntimeError('Neither a compat train binary nor a locally built train binary is available. If this is runtime-only bundle mode, the compat artifact is stale or missing.')

ensure_repo_checkout()
print('repo checkout ready at', REPO_ROOT)
print('repo mode =', repo_checkout_mode(REPO_ROOT))


In [ ]:
def maybe_install_py_package(package: str) -> None:
    try:
        __import__(package)
    except ModuleNotFoundError as err:
        raise RuntimeError(f'Missing Python package: {package}. This Kaggle notebook is configured for no-internet runs, so it will not try to pip install anything. Add the package to the image/bundle or avoid the feature that needs it.') from err

def dataset_input_file_kind(path: pathlib.Path) -> str | None:
    if not path.is_file():
        return None
    if path.name.endswith('.tar.zst') or path.name.endswith('.tar'):
        return 'tar_archive'
    if path.name.endswith('.json.gz'):
        return 'json_gz'
    if path.name.endswith('.json'):
        return 'json'
    return None


def summarize_dataset_path(path: pathlib.Path) -> dict[str, int]:
    if path.is_file():
        return {
            'tar_zst_files': int(path.name.endswith('.tar.zst') or path.name.endswith('.tar')),
            'json_files': int(path.name.endswith('.json')),
            'json_gz_files': int(path.name.endswith('.json.gz')),
        }
    return {
        'tar_zst_files': len(sorted(p for p in path.rglob('*') if p.is_file() and (p.name.endswith('.tar.zst') or p.name.endswith('.tar')))),
        'json_files': len(sorted(path.rglob('*.json'))),
        'json_gz_files': len(sorted(path.rglob('*.json.gz'))),
    }

def summarize_top_level_dataset_path(path: pathlib.Path) -> dict[str, int]:
    if path.is_file():
        return summarize_dataset_path(path)
    if not path.is_dir():
        return {'tar_zst_files': 0, 'json_files': 0, 'json_gz_files': 0}
    return {
        'tar_zst_files': len([p for p in path.iterdir() if p.is_file() and (p.name.endswith('.tar.zst') or p.name.endswith('.tar'))]),
        'json_files': len([p for p in path.iterdir() if p.is_file() and p.name.endswith('.json')]),
        'json_gz_files': len([p for p in path.iterdir() if p.is_file() and p.name.endswith('.json.gz')]),
    }

def training_input_candidate_score(path: pathlib.Path) -> tuple[int, int, int, int, str]:
    file_kind = dataset_input_file_kind(path)
    if file_kind == 'tar_archive':
        return (3, 1, 0, -len(path.parts), str(path))
    if file_kind in {'json', 'json_gz'}:
        return (2, 0, 1, -len(path.parts), str(path))
    top_level = summarize_top_level_dataset_path(path)
    if top_level['tar_zst_files'] > 1:
        return (4, top_level['tar_zst_files'], top_level['json_files'] + top_level['json_gz_files'], -len(path.parts), str(path))
    return (
        2 if top_level['tar_zst_files'] else 1,
        top_level['tar_zst_files'],
        top_level['json_files'] + top_level['json_gz_files'],
        -len(path.parts),
        str(path),
    )

def choose_best_training_input_candidate(candidates: list[pathlib.Path]) -> pathlib.Path | None:
    if not candidates:
        return None
    return max(candidates, key=training_input_candidate_score)

def discover_training_input_candidates(path: pathlib.Path) -> list[pathlib.Path]:
    if not path.exists():
        return []
    file_kind = dataset_input_file_kind(path)
    if file_kind is not None:
        return [path.resolve()]
    if not path.is_dir():
        return []
    if any(summarize_top_level_dataset_path(path).values()):
        return [path.resolve()]
    candidates: list[pathlib.Path] = []
    seen: set[str] = set()

    def add_candidate(candidate_path: pathlib.Path) -> None:
        resolved = candidate_path.resolve()
        key = str(resolved)
        if key in seen:
            return
        seen.add(key)
        candidates.append(resolved)

    for child_dir in sorted(p.resolve() for p in path.rglob('*') if p.is_dir()):
        top_level = summarize_top_level_dataset_path(child_dir)
        if top_level['tar_zst_files'] > 1:
            add_candidate(child_dir)
    for archive_path in sorted(p for p in path.rglob('*') if p.is_file() and (p.name.endswith('.tar.zst') or p.name.endswith('.tar'))):
        add_candidate(archive_path)
    for pattern in ('*.json', '*.json.gz'):
        for data_path in sorted(path.rglob(pattern)):
            parent = data_path.parent.resolve()
            if not any(summarize_top_level_dataset_path(parent).values()):
                continue
            add_candidate(parent)
    return candidates

def detect_kaggle_dataset_dir() -> pathlib.Path | None:
    if not KAGGLE_INPUT_ROOT.exists():
        return None
    candidates: list[pathlib.Path] = []
    seen: set[str] = set()
    for child in sorted(KAGGLE_INPUT_ROOT.iterdir()):
        if not child.is_dir():
            continue
        if child.name == OFFLINE_BUNDLE_DATASET_NAME or child.name in OFFLINE_BUNDLE_FALLBACK_DATASET_NAMES:
            continue
        for candidate in discover_training_input_candidates(child):
            key = str(candidate.resolve())
            if key in seen:
                continue
            seen.add(key)
            candidates.append(candidate.resolve())
    return choose_best_training_input_candidate(candidates)

def resolve_training_input(path: pathlib.Path) -> pathlib.Path:
    file_kind = dataset_input_file_kind(path)
    if file_kind is not None:
        return path.resolve()
    if path.is_dir():
        top_level = summarize_top_level_dataset_path(path)
        if top_level['tar_zst_files'] > 1:
            return path.resolve()
        top_level_archives = sorted(p.resolve() for p in path.iterdir() if p.is_file() and (p.name.endswith('.tar.zst') or p.name.endswith('.tar')))
        if len(top_level_archives) == 1:
            selected_archive = top_level_archives[0]
            print('resolved single top-level archive from', path, 'to', selected_archive)
            return selected_archive
        candidates = discover_training_input_candidates(path)
        selected = choose_best_training_input_candidate(candidates)
        if selected is not None:
            if selected.resolve() != path.resolve():
                print('resolved nested dataset input from', path, 'to', selected)
            return selected.resolve()
        recursive = summarize_dataset_path(path)
        if any(recursive.values()):
            raise RuntimeError(
                f"dataset path {path} contains MJAI/archive files, but the notebook could not find an inner directory or archive path with top-level training files. Point the notebook at the exact directory or archive that directly contains the training files."
            )
    raise RuntimeError(f'could not resolve a Hydra training dataset input from: {path}')

def activate_dataset_source(path: pathlib.Path) -> None:
    global ACTIVE_DATASET_HOST_PATH
    ACTIVE_DATASET_HOST_PATH = resolve_training_input(path)

kaggle_dataset_dir = detect_kaggle_dataset_dir()
if kaggle_dataset_dir is not None:
    print('detected Kaggle dataset dir   =', kaggle_dataset_dir)
    activate_dataset_source(kaggle_dataset_dir)
else:
    activate_dataset_source(DATASET_PATH)

print('active dataset host path      =', ACTIVE_DATASET_HOST_PATH)
print('dataset summary               =', summarize_dataset_path(ACTIVE_DATASET_HOST_PATH))


## Startup diagnostics
This prints the important environment facts up front so you can see what branch the notebook is going to take before you waste quota on preflight or training.

In [ ]:
def collect_startup_diagnostics() -> dict[str, object]:
    diagnostics: dict[str, object] = {}
    diagnostics['is_kaggle'] = IS_KAGGLE
    diagnostics['offline_bundle_dataset_name'] = OFFLINE_BUNDLE_DATASET_NAME
    diagnostics['cargo_path'] = shutil.which('cargo')
    diagnostics['python_executable'] = sys.executable
    diagnostics['repo_root'] = str(REPO_ROOT)
    diagnostics['repo_mode'] = repo_checkout_mode(REPO_ROOT)
    diagnostics['dataset_path'] = str(ACTIVE_DATASET_HOST_PATH)
    diagnostics['bundled_repo_found'] = str(detect_kaggle_repo_bundle()) if KAGGLE_INPUT_ROOT.exists() else None
    diagnostics['bundle_manifest_exists'] = bundle_manifest_path(REPO_ROOT).exists()
    diagnostics['runtime_manifest_exists'] = runtime_manifest_path(REPO_ROOT).exists()
    diagnostics['compat_binary_exists'] = compat_artifact_bin_path().exists()
    diagnostics['compat_lib_dir_exists'] = compat_artifact_lib_dir().exists()
    diagnostics['built_binary_exists'] = (REPO_ROOT / 'target' / 'release' / 'train').exists()
    try:
        import torch
        diagnostics['torch_import'] = True
        diagnostics['torch_version'] = getattr(torch, '__version__', 'unknown')
        diagnostics['torch_cuda_available'] = bool(torch.cuda.is_available())
        diagnostics['torch_file'] = getattr(torch, '__file__', 'unknown')
        torch_root = pathlib.Path(torch.__file__).resolve().parent
        diagnostics['torch_candidate_lib_dirs'] = [
            str(torch_root / 'lib'),
            str(torch_root.parent / 'lib'),
        ]
    except Exception as err:
        diagnostics['torch_import'] = False
        diagnostics['torch_error'] = repr(err)
    if compat_artifact_bin_path().exists():
        diagnostics['expected_binary_strategy'] = 'compat_artifact_binary'
    elif diagnostics['cargo_path']:
        diagnostics['expected_binary_strategy'] = 'build_on_kaggle_first'
    else:
        diagnostics['expected_binary_strategy'] = 'blocked_no_cargo_no_binary'
    return diagnostics

def detect_runtime_blockers() -> list[str]:
    blockers: list[str] = []
    cargo_path = shutil.which('cargo')
    compat_binary = compat_artifact_bin_path()
    compat_lib_dir = compat_artifact_lib_dir()
    built_binary = REPO_ROOT / 'target' / 'release' / 'train'
    if compat_binary.exists() and compat_binary.is_file():
        probe_binary = compat_binary
    elif built_binary.exists() and built_binary.is_file():
        probe_binary = built_binary
    else:
        probe_binary = compat_binary
    if cargo_path is None and probe_binary.exists() and probe_binary.is_file():
        lib_dirs: list[pathlib.Path] = []
        seen: set[str] = set()
        if compat_lib_dir.exists():
            resolved = compat_lib_dir.resolve()
            seen.add(str(resolved))
            lib_dirs.append(resolved)
        try:
            import torch
            torch_root = pathlib.Path(torch.__file__).resolve().parent
            for candidate in [torch_root / 'lib', torch_root.parent / 'lib']:
                if not (candidate / 'libtorch_cpu.so').exists():
                    continue
                resolved = candidate.resolve()
                key = str(resolved)
                if key in seen:
                    continue
                seen.add(key)
                lib_dirs.append(resolved)
        except Exception:
            pass
        for candidate in [
            pathlib.Path('/usr/local/cuda/lib64'),
            pathlib.Path('/usr/local/cuda/targets/x86_64-linux/lib'),
            pathlib.Path('/usr/local/nvidia/lib64'),
            pathlib.Path('/lib/x86_64-linux-gnu'),
            pathlib.Path('/usr/lib/x86_64-linux-gnu'),
        ]:
            if not candidate.exists():
                continue
            resolved = candidate.resolve()
            key = str(resolved)
            if key in seen:
                continue
            seen.add(key)
            lib_dirs.append(resolved)
        export_prefix = ''
        if lib_dirs:
            joined = ':'.join(str(path) for path in lib_dirs)
            export_prefix = f'export LD_LIBRARY_PATH={shlex.quote(joined)}:${{LD_LIBRARY_PATH:-}} && '
        result = subprocess.run(['bash', '-lc', f'{export_prefix}ldd {shlex.quote(str(probe_binary))} || true'], text=True, capture_output=True)
        ldd_output = (result.stdout or '') + (result.stderr or '')
        if any(token in ldd_output for token in ['GLIBC_2.38', 'GLIBC_2.39', 'GLIBCXX_3.4.32', 'CXXABI_1.3.15']):
            blockers.append('Selected train binary requires newer glibc/libstdc++ than this Kaggle image provides.')
        unresolved = []
        for line in ldd_output.splitlines():
            if '=> not found' not in line:
                continue
            unresolved.append(line.split('=>', 1)[0].strip())
        if unresolved:
            blockers.append('Selected train binary has unresolved shared-library dependencies from the current runtime loader path: ' + ', '.join(unresolved))
        if blockers:
            blockers.append('Cargo is not available, so Kaggle cannot rebuild train locally to escape the ABI mismatch.')
    return blockers

STARTUP_DIAGNOSTICS = collect_startup_diagnostics()
RUNTIME_BLOCKERS = detect_runtime_blockers()
print(json.dumps(STARTUP_DIAGNOSTICS, indent=2))
if RUNTIME_BLOCKERS:
    print('\nRUNTIME BLOCKERS DETECTED:')
    for blocker in RUNTIME_BLOCKERS:
        print('-', blocker)


In [ ]:
def write_baseline_config(path: pathlib.Path) -> pathlib.Path:
    candidate_list = ', '.join(str(x) for x in PREFLIGHT_CANDIDATES)
    data_dir = str(ACTIVE_DATASET_HOST_PATH)
    output_dir = str(OUTPUT_DIR)
    yaml_text = textwrap.dedent(f'''\
    data_dir: {data_dir}
    output_dir: {output_dir}
    num_epochs: {NUM_EPOCHS}
    batch_size: {BATCH_SIZE}
    train_fraction: {TRAIN_FRACTION}
    microbatch_size: {TRAIN_MICROBATCH_SIZE if TRAIN_MICROBATCH_SIZE is not None else 'null'}
    validation_microbatch_size: {VALIDATION_MICROBATCH_SIZE if VALIDATION_MICROBATCH_SIZE is not None else 'null'}
    augment: {'true' if AUGMENT else 'false'}
    seed: {SEED}
    device: {DEVICE}
    num_threads: {NUM_THREADS if NUM_THREADS is not None else 'null'}
    buffer_games: {BUFFER_GAMES}
    buffer_samples: {BUFFER_SAMPLES}
    tensorboard: {'true' if TENSORBOARD else 'false'}
    archive_queue_bound: {ARCHIVE_QUEUE_BOUND}
    validation_every_n_epochs: 1
    max_skip_logs_per_source: {MAX_SKIP_LOGS_PER_SOURCE}
    log_every_n_steps: {LOG_EVERY_N_STEPS}
    validate_every_n_steps: {VALIDATE_EVERY_N_STEPS}
    checkpoint_every_n_steps: {CHECKPOINT_EVERY_N_STEPS}
    max_validation_samples: {MAX_VALIDATION_SAMPLES}
    preflight:
      warmup_steps: 2
      measure_steps: 3
      required_successes: 2
      candidate_microbatches: [{candidate_list}]
      local_refinement_enabled: true
      finalist_extra_successes: 1
      finalist_extra_measure_steps: 2
      search_coordinate_rounds: 2
      search_top_k: 3
      rl_probe_min_free_memory_bytes: 2147483648
      rl_probe_memory_headroom_ratio: 0.05
      rl_probe_growth_safety_factor: 1.35
    ''').strip() + '\n'
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml_text)
    return path

write_baseline_config(CONFIG_PATH)
print(CONFIG_PATH.read_text())

def detect_torch_lib_dirs() -> list[pathlib.Path]:
    try:
        import torch
    except Exception:
        return []
    torch_root = pathlib.Path(torch.__file__).resolve().parent
    candidates = [torch_root / 'lib', torch_root.parent / 'lib']
    nvidia_root = torch_root.parent / 'nvidia'
    if nvidia_root.exists():
        candidates.extend(sorted(nvidia_root.glob('*/lib')))
    return [path for path in candidates if path.exists()]

def detect_cuda_lib_dirs() -> list[pathlib.Path]:
    candidates = [
        pathlib.Path('/usr/local/cuda/lib64'),
        pathlib.Path('/usr/local/cuda/targets/x86_64-linux/lib'),
        pathlib.Path('/usr/local/nvidia/lib64'),
        pathlib.Path('/lib/x86_64-linux-gnu'),
        pathlib.Path('/usr/lib/x86_64-linux-gnu'),
    ]
    return [path for path in candidates if path.exists()]

def compat_metadata_filenames() -> list[str]:
    return [
        'runtime-manifest.json',
        'ldd-train.txt',
        'ldd-train-summary.txt',
        'abi-symbols.txt',
        'lib-summary.txt',
        'lib-manifest.tsv',
    ]

def sha256_file(path: pathlib.Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def ordered_unique_existing_paths(paths: list[pathlib.Path]) -> list[pathlib.Path]:
    ordered: list[pathlib.Path] = []
    seen: set[str] = set()
    for path in paths:
        if not path.exists():
            continue
        resolved = path.resolve()
        key = str(resolved)
        if key in seen:
            continue
        seen.add(key)
        ordered.append(resolved)
    return ordered

def runtime_loader_details() -> dict[str, Any]:
    compat_lib = compat_artifact_lib_dir()
    torch_lib_dirs = detect_torch_lib_dirs()
    cuda_lib_dirs = detect_cuda_lib_dirs()
    candidate_dirs = [compat_lib, *torch_lib_dirs, *cuda_lib_dirs]
    relevant_dirs: list[pathlib.Path] = []
    seen_relevant: set[str] = set()
    for path in candidate_dirs:
        rendered = str(path.resolve()) if path.exists() else str(path)
        if rendered in seen_relevant:
            continue
        seen_relevant.add(rendered)
        relevant_dirs.append(path)
    ordered_loader_dirs = ordered_unique_existing_paths(candidate_dirs)
    loader_path_prefix = ':'.join(str(path) for path in ordered_loader_dirs)
    inherited_ld_library_path = os.environ.get('LD_LIBRARY_PATH', '')
    if loader_path_prefix and inherited_ld_library_path:
        final_ld_library_path = f'{loader_path_prefix}:{inherited_ld_library_path}'
    else:
        final_ld_library_path = loader_path_prefix or inherited_ld_library_path
    runtime_manifest = load_runtime_manifest()
    sentinel_libs = [
        str(name)
        for name in (runtime_manifest or {}).get('required_runtime_sentinels', [
            'libtorch_cpu.so',
            'libtorch_cuda.so',
            'libc10.so',
            'libc10_cuda.so',
            'libcusparseLt.so.0',
        ])
    ]
    sentinel_presence = {
        lib_name: {str(path): (path / lib_name).exists() for path in relevant_dirs}
        for lib_name in sentinel_libs
    }
    return {
        'repo_mode': repo_checkout_mode(REPO_ROOT),
        'compat_lib_dir': str(compat_lib),
        'torch_lib_dirs': [str(path) for path in torch_lib_dirs],
        'cuda_lib_dirs': [str(path) for path in cuda_lib_dirs],
        'loader_candidate_dirs': [str(path) for path in relevant_dirs],
        'final_ordered_loader_dirs': [str(path) for path in ordered_loader_dirs],
        'loader_path_prefix': loader_path_prefix,
        'inherited_ld_library_path': inherited_ld_library_path,
        'final_ld_library_path': final_ld_library_path,
        'sentinel_presence': sentinel_presence,
    }

def build_loader_export_prefix(loader_details: dict[str, Any]) -> str:
    loader_path_prefix = str(loader_details['loader_path_prefix'])
    if not loader_path_prefix:
        return ''
    return f'export LD_LIBRARY_PATH={shlex.quote(loader_path_prefix)}:${{LD_LIBRARY_PATH:-}} && '

def selected_train_binary_path() -> pathlib.Path:
    compat_binary = compat_artifact_bin_path()
    built_binary = REPO_ROOT / 'target' / 'release' / 'train'
    if compat_binary.exists() and compat_binary.is_file():
        return compat_binary
    if built_binary.exists() and built_binary.is_file():
        return built_binary
    if repo_checkout_mode(REPO_ROOT) == 'full_repo' and shutil.which('cargo') is not None:
        return built_binary
    return compat_binary

def build_help_smoke_command(binary: pathlib.Path, loader_details: dict[str, Any]) -> list[str]:
    quoted_binary = shlex.quote(str(binary))
    export_prefix = build_loader_export_prefix(loader_details)
    shell_cmd = f'{export_prefix}if [ -e {quoted_binary} ]; then chmod +x {quoted_binary}; fi; {quoted_binary} --help'
    return ['bash', '-lc', shell_cmd]

def compat_metadata_file_status(filename: str) -> dict[str, Any]:
    path = compat_metadata_path(filename)
    return {
        'path': str(path),
        'exists': path.exists(),
        'size_bytes': path.stat().st_size if path.exists() and path.is_file() else 0,
        'sha256': sha256_file(path) if path.exists() and path.is_file() else None,
    }

def checkout_manifest_status() -> dict[str, Any]:
    bundle_manifest = load_bundle_manifest(REPO_ROOT)
    runtime_manifest = load_runtime_manifest(REPO_ROOT)
    bundle_errors = validate_bundle_manifest_contract(REPO_ROOT, bundle_manifest) if bundle_manifest is not None else []
    runtime_errors = validate_runtime_manifest_contract(REPO_ROOT, runtime_manifest) if runtime_manifest is not None else []
    payload_path = detect_kaggle_repo_bundle_payload() if KAGGLE_INPUT_ROOT.exists() else None
    payload_manifest = None
    payload_error = None
    if payload_path is not None:
        try:
            payload_manifest, payload_manifest_name = load_payload_bundle_manifest(payload_path)
        except Exception as err:
            payload_manifest_name = None
            payload_error = repr(err)
        else:
            payload_error = None
    else:
        payload_manifest_name = None
    return {
        'repo_mode': repo_checkout_mode(REPO_ROOT),
        'bundle_manifest_path': str(bundle_manifest_path(REPO_ROOT)),
        'bundle_manifest_exists': bundle_manifest is not None,
        'bundle_manifest_required_file_count': len((bundle_manifest or {}).get('required_files', [])),
        'bundle_manifest_errors': bundle_errors,
        'runtime_manifest_path': str(runtime_manifest_path(REPO_ROOT)),
        'runtime_manifest_exists': runtime_manifest is not None,
        'runtime_manifest_required_lib_count': len((runtime_manifest or {}).get('required_runtime_libraries', [])),
        'runtime_manifest_errors': runtime_errors,
        'payload_bundle_path': str(payload_path) if payload_path is not None else None,
        'payload_bundle_manifest_name': payload_manifest_name,
        'payload_bundle_manifest_error': payload_error,
        'payload_matches_checkout_bundle_manifest': canonical_manifest_text(payload_manifest) == canonical_manifest_text(bundle_manifest) if payload_manifest is not None and bundle_manifest is not None else None,
    }

def validate_runtime_launch_ready() -> list[str]:
    repo_root = ensure_repo_checkout()
    mode = repo_checkout_mode(repo_root)
    errors: list[str] = []
    if mode == 'runtime_only_bundle':
        errors.extend(validate_runtime_bundle_checkout(repo_root))
        if not compat_artifact_bin_path(repo_root).exists():
            errors.append(f'runtime-only bundle is missing compat train binary at {compat_artifact_bin_path(repo_root)}')
    elif mode == 'full_repo':
        errors.extend(validate_full_repo_checkout(repo_root))
        compat_binary = compat_artifact_bin_path(repo_root)
        built_binary = repo_root / 'target' / 'release' / 'train'
        if compat_binary.exists() and runtime_manifest_path(repo_root).exists():
            errors.extend(validate_runtime_manifest_contract(repo_root))
        if not compat_binary.exists() and not built_binary.exists() and shutil.which('cargo') is None:
            errors.append('full repo checkout has neither compat train runtime nor cargo available for a local build')
    else:
        errors.append(f'unknown Hydra checkout mode under {repo_root}')
    deduped: list[str] = []
    seen: set[str] = set()
    for error in errors:
        if error in seen:
            continue
        seen.add(error)
        deduped.append(error)
    return deduped

def debug_dump_path() -> pathlib.Path:
    return NOTEBOOK_STATE_ROOT / 'debug-artifacts' / 'kaggle-runtime-debug.txt'

def debug_bundle_candidates(runtime_dump_path: pathlib.Path) -> list[pathlib.Path]:
    candidates: list[pathlib.Path] = [
        runtime_dump_path,
        CONFIG_PATH,
        bundle_manifest_path(REPO_ROOT),
        runtime_manifest_path(REPO_ROOT),
    ]
    candidates.extend(compat_metadata_path(name) for name in compat_metadata_filenames())
    candidates.extend([
        preflight_cache_path(),
        latest_resume_state_path(),
        OUTPUT_DIR / 'bc' / 'training_log.jsonl',
        OUTPUT_DIR / 'bc' / 'step_log.jsonl',
        OUTPUT_DIR / 'bc' / 'kaggle-runtime-debug.txt',
    ])
    if PERSISTENT_LOG_ROOT.exists():
        candidates.extend(sorted(path for path in PERSISTENT_LOG_ROOT.rglob('*') if path.is_file()))
    ordered: list[pathlib.Path] = []
    seen: set[str] = set()
    for path in candidates:
        if not path.exists() or not path.is_file():
            continue
        resolved = path.resolve()
        key = str(resolved)
        if key in seen:
            continue
        seen.add(key)
        ordered.append(resolved)
    return ordered

def debug_bundle_archive_name(path: pathlib.Path) -> str:
    try:
        if path.is_relative_to(REPO_ROOT):
            return f'repo/{path.relative_to(REPO_ROOT).as_posix()}'
    except Exception:
        pass
    try:
        if path.is_relative_to(NOTEBOOK_STATE_ROOT):
            return f'state/{path.relative_to(NOTEBOOK_STATE_ROOT).as_posix()}'
    except Exception:
        pass
    return f'misc/{path.name}'

def write_latest_debug_bundle() -> tuple[pathlib.Path, pathlib.Path, dict[str, Any], str, list[str]]:
    dump_path, debug_data, debug_text = write_runtime_debug_dump()
    ensure_persistent_state_dirs()
    included_paths = [str(path) for path in debug_bundle_candidates(dump_path)]
    with zipfile.ZipFile(LATEST_DEBUG_BUNDLE_PATH, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for path in debug_bundle_candidates(dump_path):
            zf.write(path, arcname=debug_bundle_archive_name(path))
    return LATEST_DEBUG_BUNDLE_PATH, dump_path, debug_data, debug_text, included_paths

def collect_runtime_debug_dump() -> dict[str, Any]:
    startup_diagnostics = collect_startup_diagnostics()
    runtime_blockers = detect_runtime_blockers()
    launch_validation_errors = validate_runtime_launch_ready()
    loader_details = runtime_loader_details()
    binary = selected_train_binary_path()
    smoke_cmd = build_help_smoke_command(binary, loader_details)
    smoke_result = subprocess.run(smoke_cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    compat_metadata_files = {
        name: compat_metadata_file_status(name)
        for name in compat_metadata_filenames()
    }
    try:
        training_snapshot = training_status_snapshot()
    except Exception as err:
        training_snapshot = {'error': repr(err)}
    try:
        hardware_snapshot = pretty_snapshot()
    except Exception as err:
        hardware_snapshot = f'<pretty_snapshot failed: {err!r}>'
    cache_path = preflight_cache_path()
    return {
        'repo_mode': repo_checkout_mode(REPO_ROOT),
        'config_path': str(CONFIG_PATH),
        'config_text': CONFIG_PATH.read_text() if CONFIG_PATH.exists() else '<missing config>',
        'startup_diagnostics': startup_diagnostics,
        'runtime_blockers': runtime_blockers,
        'launch_validation_errors': launch_validation_errors,
        'checkout_manifest_status': checkout_manifest_status(),
        'persistent_log_status': persistent_log_status_snapshot(),
        'selected_binary_path': str(binary),
        'selected_binary_exists': binary.exists() and binary.is_file(),
        'compat_lib_dir': str(compat_artifact_lib_dir()),
        'loader_details': loader_details,
        'compat_metadata_files': compat_metadata_files,
        'preflight_cache': {
            'path': str(cache_path),
            'exists': cache_path.exists(),
        },
        'training_status_snapshot': training_snapshot,
        'pretty_snapshot': hardware_snapshot,
        'latest_debug_bundle_path': str(LATEST_DEBUG_BUNDLE_PATH),
        'latest_debug_bundle_exists': LATEST_DEBUG_BUNDLE_PATH.exists(),
        'help_smoke': {
            'command': ' '.join(shlex.quote(part) for part in smoke_cmd),
            'returncode': smoke_result.returncode,
            'stdout': smoke_result.stdout,
            'stderr': smoke_result.stderr,
        },
    }

def render_runtime_debug_dump(debug_data: dict[str, Any]) -> str:
    sections = [
        'Hydra Kaggle runtime debug dump',
        '',
        '=== current config ===',
        f"path: {debug_data['config_path']}",
        str(debug_data['config_text']).rstrip() or '<empty>',
        '',
        '=== startup diagnostics ===',
        json.dumps(debug_data['startup_diagnostics'], indent=2, sort_keys=True),
        '',
        '=== runtime blockers ===',
        json.dumps(debug_data['runtime_blockers'], indent=2, sort_keys=True),
        '',
        '=== binary selection ===',
        json.dumps(
            {
                'selected_binary_path': debug_data['selected_binary_path'],
                'selected_binary_exists': debug_data['selected_binary_exists'],
                'compat_lib_dir': debug_data['compat_lib_dir'],
            },
            indent=2,
            sort_keys=True,
        ),
        '',
        '=== checkout manifest status ===',
        json.dumps(debug_data['checkout_manifest_status'], indent=2, sort_keys=True),
        '',
        '=== persistent log status ===',
        json.dumps(debug_data['persistent_log_status'], indent=2, sort_keys=True),
        '',
        '=== launch validation errors ===',
        json.dumps(debug_data['launch_validation_errors'], indent=2, sort_keys=True),
        '',
        '=== loader details ===',
        json.dumps(debug_data['loader_details'], indent=2, sort_keys=True),
        '',
        '=== compat metadata files ===',
        json.dumps(debug_data['compat_metadata_files'], indent=2, sort_keys=True),
        '',
        '=== preflight cache ===',
        json.dumps(debug_data['preflight_cache'], indent=2, sort_keys=True),
        '',
        '=== training status snapshot ===',
        json.dumps(debug_data['training_status_snapshot'], indent=2, sort_keys=True),
        '',
        '=== latest debug bundle ===',
        json.dumps(
            {
                'path': debug_data['latest_debug_bundle_path'],
                'exists': debug_data['latest_debug_bundle_exists'],
            },
            indent=2,
            sort_keys=True,
        ),
        '',
        '=== pretty snapshot ===',
        str(debug_data['pretty_snapshot']).rstrip() or '<empty>',
        '',
        '=== help smoke ===',
        json.dumps(
            {
                'command': debug_data['help_smoke']['command'],
                'returncode': debug_data['help_smoke']['returncode'],
            },
            indent=2,
            sort_keys=True,
        ),
        '',
        '--- help smoke stdout ---',
        str(debug_data['help_smoke']['stdout']).rstrip() or '<empty>',
        '',
        '--- help smoke stderr ---',
        str(debug_data['help_smoke']['stderr']).rstrip() or '<empty>',
    ]
    return '\n'.join(sections).rstrip() + '\n'

def write_runtime_debug_dump() -> tuple[pathlib.Path, dict[str, Any], str]:
    debug_data = collect_runtime_debug_dump()
    dump_path = debug_dump_path()
    dump_path.parent.mkdir(parents=True, exist_ok=True)
    debug_text = render_runtime_debug_dump(debug_data)
    dump_path.write_text(debug_text)
    return dump_path, debug_data, debug_text

def debug_dump_preview(debug_text: str, max_lines: int = 32) -> str:
    lines = debug_text.splitlines()
    preview = '\n'.join(lines[:max_lines])
    if len(lines) > max_lines:
        preview = f"{preview}\n... ({len(lines) - max_lines} more lines)"
    return preview

def active_train_command(*extra_args: str) -> list[str]:
    launch_validation_errors = validate_runtime_launch_ready()
    if launch_validation_errors:
        raise RuntimeError('Hydra runtime is not launch-ready in this session. ' + ' '.join(launch_validation_errors))
    runtime_blockers = detect_runtime_blockers()
    if runtime_blockers:
        joined = ' '.join(runtime_blockers)
        raise RuntimeError(f'Kaggle runtime is not compatible with Hydra execution in this session. {joined}')
    binary = ensure_native_train_binary()
    quoted = ' '.join(shlex.quote(part) for part in [str(binary), str(CONFIG_PATH), *extra_args])
    export_prefix = build_loader_export_prefix(runtime_loader_details())
    return ['bash', '-lc', f'{export_prefix}chmod +x {shlex.quote(str(binary))} && exec {quoted}']

def read_json_lines(path: pathlib.Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))
    return rows


## Notebook UI: preflight tuning
This is the click-driven control panel for dataset selection, config writing, preflight, and optional application of the selected runtime back into the YAML.

In [ ]:
def maybe_install_widgets() -> None:
    maybe_install_py_package('ipywidgets')

def list_dataset_candidates() -> list[pathlib.Path]:
    candidates: list[pathlib.Path] = []
    seen: set[str] = set()

    def add_candidate(path: pathlib.Path) -> None:
        resolved = path.resolve()
        key = str(resolved)
        if key in seen:
            return
        seen.add(key)
        candidates.append(resolved)

    add_candidate(ACTIVE_DATASET_HOST_PATH)
    if DATASET_PATH.exists():
        add_candidate(resolve_training_input(DATASET_PATH))
    if KAGGLE_INPUT_ROOT.exists():
        for child in sorted(KAGGLE_INPUT_ROOT.iterdir()):
            if not child.is_dir():
                continue
            if child.name == OFFLINE_BUNDLE_DATASET_NAME or child.name in OFFLINE_BUNDLE_FALLBACK_DATASET_NAMES:
                continue
            for candidate in discover_training_input_candidates(child):
                add_candidate(candidate)
    return candidates

def preflight_runtime_smoke_check() -> None:
    launch_validation_errors = validate_runtime_launch_ready()
    if launch_validation_errors:
        raise RuntimeError('Hydra runtime is not launch-ready in this session:\n' + '\n'.join(f'- {error}' for error in launch_validation_errors))
    binary = ensure_native_train_binary()
    cmd = build_help_smoke_command(binary, runtime_loader_details())
    result = subprocess.run(cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    stderr = result.stderr or ''
    combined_output = (result.stdout or '') + stderr
    if any(token in combined_output for token in ['error while loading shared libraries', 'GLIBC_', 'GLIBCXX_', 'CXXABI_', 'cannot open shared object file', 'No such file or directory']):
        raise RuntimeError(f"pre-launch runtime smoke check failed for {binary}\nSTDOUT:\n{result.stdout or '<empty>'}\nSTDERR:\n{stderr or '<empty>'}")

def build_preflight_ui():
    maybe_install_widgets()
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    dataset_candidates = list_dataset_candidates()
    dataset_dropdown = widgets.Dropdown(
        options=[(str(path), str(path)) for path in dataset_candidates] or [('(current dataset path)', str(DATASET_PATH))],
        value=str(ACTIVE_DATASET_HOST_PATH),
        description='Dataset:',
        layout=widgets.Layout(width='95%'),
    )
    dataset_help = widgets.HTML(value='<b>Dataset path:</b> training and preflight use exactly the resolved path shown here when the config is written. A directory means train on all top-level MJAI/archive files inside it; an archive path means train on just that one archive.')
    batch_size_widget = widgets.IntText(value=BATCH_SIZE, description='Batch size:')
    epochs_widget = widgets.IntText(value=NUM_EPOCHS, description='Epochs:')
    train_fraction_widget = widgets.FloatSlider(value=TRAIN_FRACTION, min=0.5, max=0.99, step=0.01, description='Train frac:')
    train_fraction_help = widgets.HTML()
    tensorboard_widget = widgets.Checkbox(value=TENSORBOARD, description='TensorBoard')
    augment_widget = widgets.Checkbox(value=AUGMENT, description='Augment')
    write_config_btn = widgets.Button(description='Write config', button_style='info')
    preflight_btn = widgets.Button(description='Launch preflight', button_style='warning')
    poll_preflight_btn = widgets.Button(description='Poll preflight', button_style='info')
    stop_preflight_btn = widgets.Button(description='Stop preflight', button_style='warning')
    debug_dump_btn = widgets.Button(description='Dump latest logs', button_style='info')
    apply_btn = widgets.Button(description='Apply preflight runtime', button_style='success')
    auto_refresh_widget = widgets.Checkbox(value=False, description='Auto-refresh while running')
    refresh_seconds_widget = widgets.IntSlider(value=10, min=3, max=60, step=1, description='Refresh s:')
    output = widgets.Output()
    status_html = widgets.HTML(value='<b>Preflight status:</b> idle')

    def render_train_fraction_help() -> None:
        value = float(train_fraction_widget.value)
        validation_fraction = max(0.0, 1.0 - value)
        train_fraction_help.value = (
            f"<b>Train/validation split:</b> about {value:.0%} train / {validation_fraction:.0%} validation. "
            "This is a dataset split knob, not a progress or time knob."
        )

    def refresh_globals_from_widgets():
        global ACTIVE_DATASET_HOST_PATH, BATCH_SIZE, NUM_EPOCHS, TRAIN_FRACTION, TENSORBOARD, AUGMENT
        ACTIVE_DATASET_HOST_PATH = pathlib.Path(dataset_dropdown.value)
        activate_dataset_source(ACTIVE_DATASET_HOST_PATH)
        BATCH_SIZE = int(batch_size_widget.value)
        NUM_EPOCHS = int(epochs_widget.value)
        TRAIN_FRACTION = float(train_fraction_widget.value)
        TENSORBOARD = bool(tensorboard_widget.value)
        AUGMENT = bool(augment_widget.value)

    def on_write_config(_):
        with output:
            clear_output(wait=True)
            refresh_globals_from_widgets()
            ensure_repo_checkout()
            preflight_runtime_smoke_check()
            write_baseline_config(CONFIG_PATH)
            print('wrote config to', CONFIG_PATH)
            print(f'train/validation split ≈ {TRAIN_FRACTION:.0%}/{1.0 - TRAIN_FRACTION:.0%}')
            print(CONFIG_PATH.read_text())

    def render_preflight_status() -> None:
        status = preflight_status_snapshot()
        lines = [
            f"<b>Running:</b> {status['running']}",
            f"<b>PID:</b> {status['pid']}",
            f"<b>Return code:</b> {status['returncode']}",
            f"<b>Status source:</b> {status['status_source']}",
            f"<b>Cache exists:</b> {status['preflight_cache_exists']}",
            f"<b>Cache loaded:</b> {status['preflight_cache_loaded']}",
            f"<b>Selected runtime present:</b> {status['selected_runtime_present']}",
        ]
        stop_signal = status.get('process_state', {}).get('stop_signal')
        if stop_signal is not None:
            lines.append(f"<b>Last stop signal:</b> {stop_signal}")
        status_html.value = '<br>'.join(lines)

    def refresh_preflight_view(max_lines: int = 200) -> None:
        render_preflight_status()
        print(json.dumps(preflight_status_snapshot(), indent=2))
        logs = drain_preflight_output(max_lines=max_lines)
        if logs:
            print('\n'.join(logs))
        else:
            print('no new preflight log lines')
        cache = load_preflight_cache()
        if cache is not None:
            print('\npreflight cache loaded = True')
            print(json.dumps(cache, indent=2))

    def on_run_preflight(_):
        with output:
            clear_output(wait=True)
            refresh_globals_from_widgets()
            ensure_repo_checkout()
            preflight_runtime_smoke_check()
            write_baseline_config(CONFIG_PATH)
            start_preflight_process()
            render_preflight_status()
            print('preflight launched in background')
            print(f'train/validation split ≈ {TRAIN_FRACTION:.0%}/{1.0 - TRAIN_FRACTION:.0%}')
            logs = drain_preflight_output(max_lines=40)
            if logs:
                print('\n'.join(logs))

    def on_poll_preflight(_):
        with output:
            clear_output(wait=True)
            refresh_preflight_view(max_lines=200)

    def on_stop_preflight(_):
        with output:
            clear_output(wait=True)
            stop_preflight_process()
            render_preflight_status()
            print(json.dumps(preflight_status_snapshot(), indent=2))

    def on_auto_refresh_preflight(change):
        if not change['new']:
            return
        start_auto_refresh(
            'preflight',
            lambda: None,
            lambda: preflight_status_snapshot()['running'],
            int(refresh_seconds_widget.value),
        )
        with output:
            clear_output(wait=True)
            print('preflight auto-refresh enabled; use Poll preflight for an immediate snapshot. Interrupting this cell does not stop the subprocess.')

    def on_dump_latest_logs(_):
        with output:
            clear_output(wait=True)
            refresh_globals_from_widgets()
            ensure_repo_checkout()
            write_baseline_config(CONFIG_PATH)
            bundle_path, dump_path, debug_data, debug_text, included_paths = write_latest_debug_bundle()
            summary = {
                'repo_mode': debug_data['repo_mode'],
                'selected_binary_path': debug_data['selected_binary_path'],
                'selected_binary_exists': debug_data['selected_binary_exists'],
                'final_ordered_loader_dirs': debug_data['loader_details']['final_ordered_loader_dirs'],
                'runtime_blockers': debug_data['runtime_blockers'],
                'launch_validation_errors': debug_data['launch_validation_errors'],
                'smoke_returncode': debug_data['help_smoke']['returncode'],
                'preflight_cache_exists': debug_data['preflight_cache']['exists'],
                'bundle_manifest_exists': debug_data['checkout_manifest_status']['bundle_manifest_exists'],
                'runtime_manifest_exists': debug_data['checkout_manifest_status']['runtime_manifest_exists'],
                'persistent_logs_root': str(PERSISTENT_LOG_ROOT),
                'latest_debug_bundle_path': str(bundle_path),
                'debug_bundle_included_files': included_paths,
            }
            print('wrote config to', CONFIG_PATH)
            print('saved runtime debug dump to', dump_path)
            print('saved latest debug bundle to', bundle_path)
            print(json.dumps(summary, indent=2, sort_keys=True))
            print('\n=== debug preview ===')
            print(debug_dump_preview(debug_text))
            print('=== end debug preview ===')

    def on_apply_runtime(_):
        with output:
            clear_output(wait=True)
            apply_preflight_runtime_to_config()
            print(CONFIG_PATH.read_text())

    train_fraction_widget.observe(lambda change: render_train_fraction_help(), names='value')
    write_config_btn.on_click(wrap_widget_callback('preflight_write_config', on_write_config))
    preflight_btn.on_click(wrap_widget_callback('preflight_launch', on_run_preflight))
    poll_preflight_btn.on_click(wrap_widget_callback('preflight_poll_status', on_poll_preflight))
    stop_preflight_btn.on_click(wrap_widget_callback('preflight_stop', on_stop_preflight))
    auto_refresh_widget.observe(wrap_widget_callback('preflight_auto_refresh_toggle', on_auto_refresh_preflight), names='value')
    debug_dump_btn.on_click(wrap_widget_callback('preflight_dump_latest_logs', on_dump_latest_logs))
    apply_btn.on_click(wrap_widget_callback('preflight_apply_runtime', on_apply_runtime))

    controls = widgets.VBox([
        dataset_dropdown,
        dataset_help,
        widgets.HBox([batch_size_widget, epochs_widget]),
        train_fraction_widget,
        train_fraction_help,
        widgets.HBox([tensorboard_widget, augment_widget]),
        widgets.HBox([write_config_btn, preflight_btn, poll_preflight_btn, stop_preflight_btn]),
        widgets.HBox([debug_dump_btn, apply_btn]),
        widgets.HBox([auto_refresh_widget, refresh_seconds_widget]),
        status_html,
        output,
    ])
    display(controls)
    render_train_fraction_help()
    render_preflight_status()


## Notebook UI: training launch
This is the separate click-driven launcher for actual training. Use it after preflight is done and the config looks good.

In [ ]:
def latest_metrics_summary() -> dict[str, Any]:
    summary: dict[str, Any] = {}
    step_rows = read_json_lines(OUTPUT_DIR / 'bc' / 'step_log.jsonl')
    training_rows = read_json_lines(OUTPUT_DIR / 'bc' / 'training_log.jsonl')
    if step_rows:
        last = step_rows[-1]
        for key in [
            'global_step',
            'epoch',
            'lr',
            'train_total_loss',
            'train_policy_agreement',
            'val_total_loss',
            'val_policy_loss',
            'val_policy_agreement',
            'best_val_policy_loss',
            'best_val_agreement',
        ]:
            if key in last:
                summary[key] = last[key]
    if training_rows:
        summary['completed_epochs'] = training_rows[-1].get('epoch')
    return summary

def build_training_ui():
    maybe_install_widgets()
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    resume_widget = widgets.Checkbox(value=False, description='Resume from latest checkpoint')
    inline_tb_widget = widgets.Checkbox(value=IS_KAGGLE, description='Show inline TensorBoard after launch')
    dashboard_widget = widgets.Checkbox(value=True, description='Plot JSONL dashboard after launch')
    launch_btn = widgets.Button(description='Launch training', button_style='danger')
    poll_btn = widgets.Button(description='Poll status', button_style='info')
    stop_btn = widgets.Button(description='Stop training', button_style='warning')
    auto_refresh_widget = widgets.Checkbox(value=False, description='Auto-refresh while running')
    refresh_seconds_widget = widgets.IntSlider(value=10, min=3, max=60, step=1, description='Refresh s:')
    show_tb_btn = widgets.Button(description='Show inline TensorBoard', button_style='info')
    show_dash_btn = widgets.Button(description='Plot dashboard', button_style='success')
    dump_logs_btn = widgets.Button(description='Dump latest logs', button_style='info')
    bundle_artifacts_btn = widgets.Button(description='Bundle artifacts', button_style='info')
    output = widgets.Output()
    summary_html = widgets.HTML(value='<b>No metrics yet.</b>')
    tensorboard_html = widgets.HTML(value='<b>TensorBoard:</b> no run yet')

    def render_summary_card() -> None:
        status = training_status_snapshot()
        metrics = latest_metrics_summary()
        lines = [
            f"<b>Running:</b> {status['running']}",
            f"<b>PID:</b> {status['pid']}",
            f"<b>Return code:</b> {status['returncode']}",
            f"<b>Status source:</b> {status['status_source']}",
        ]
        stop_signal = status.get('process_state', {}).get('stop_signal')
        if stop_signal is not None:
            lines.append(f"<b>Last stop signal:</b> {stop_signal}")
        for key, value in metrics.items():
            lines.append(f"<b>{key}:</b> {value}")
        summary_html.value = '<br>'.join(lines)

    def render_tensorboard_status() -> None:
        status = tensorboard_status_snapshot()
        lines = [
            f"<b>Logdir:</b> {status['logdir']}",
            f"<b>Event file count:</b> {status['event_file_count']}",
            f"<b>Inline available:</b> {status['inline_available']}",
        ]
        if status['tensorboard_error'] is not None:
            lines.append(f"<b>TensorBoard import:</b> {status['tensorboard_error']}")
        tensorboard_html.value = '<br>'.join(lines)

    def refresh_training_view(max_lines: int = 200) -> None:
        print(json.dumps(training_status_snapshot(), indent=2))
        render_summary_card()
        render_tensorboard_status()
        print('\n=== hardware snapshot ===')
        print(pretty_snapshot())
        print('=== end hardware snapshot ===\n')
        logs = drain_training_output(max_lines=max_lines)
        if logs:
            print('\n'.join(logs))
        else:
            print('no new log lines')

    def on_launch(_):
        with output:
            clear_output(wait=True)
            ensure_repo_checkout()
            preflight_runtime_smoke_check()
            if resume_widget.value:
                configure_resume_from_latest_checkpoint()
            else:
                clear_resume_from_config()
            start_training_process()
            print('training launched in background')
            print(json.dumps(training_status_snapshot(), indent=2))
            render_summary_card()
            render_tensorboard_status()
            logs = drain_training_output(max_lines=40)
            if logs:
                print('\n'.join(logs))
            if inline_tb_widget.value:
                print('inline TensorBoard will work once event files exist')
            if dashboard_widget.value:
                print('use Poll status or Plot dashboard to refresh charts while training runs')

    def on_poll(_):
        with output:
            clear_output(wait=True)
            refresh_training_view(max_lines=200)

    def on_stop(_):
        with output:
            clear_output(wait=True)
            stop_training_process()
            print(json.dumps(training_status_snapshot(), indent=2))
            render_summary_card()
            render_tensorboard_status()

    def on_auto_refresh(change):
        if not change['new']:
            return
        start_auto_refresh(
            'training',
            lambda: None,
            lambda: training_status_snapshot()['running'],
            int(refresh_seconds_widget.value),
        )
        with output:
            clear_output(wait=True)
            print('training auto-refresh enabled; use Poll status for an immediate snapshot. Interrupting this cell does not stop the subprocess.')

    def on_show_tb(_):
        with output:
            clear_output(wait=True)
            render_tensorboard_status()
            try:
                show_inline_tensorboard()
            except Exception as err:
                bundle_path = write_training_artifact_bundle()
                print(f'inline TensorBoard unavailable: {err!r}')
                print('offline fallback: use Plot dashboard and/or download training artifacts from', bundle_path)

    def on_show_dash(_):
        with output:
            clear_output(wait=True)
            plot_bc_dashboard()

    def on_dump_latest_logs(_):
        with output:
            clear_output(wait=True)
            ensure_repo_checkout()
            bundle_path, dump_path, debug_data, debug_text, included_paths = write_latest_debug_bundle()
            print('saved runtime debug dump to', dump_path)
            print('saved latest debug bundle to', bundle_path)
            print(json.dumps({
                'repo_mode': debug_data['repo_mode'],
                'training_running': debug_data['training_status_snapshot'].get('running'),
                'launch_validation_errors': debug_data['launch_validation_errors'],
                'latest_debug_bundle_path': str(bundle_path),
                'debug_bundle_included_files': included_paths,
            }, indent=2, sort_keys=True))
            print('\n=== debug preview ===')
            print(debug_dump_preview(debug_text))
            print('=== end debug preview ===')

    def on_bundle_training_artifacts(_):
        with output:
            clear_output(wait=True)
            bundle_path = write_training_artifact_bundle()
            print('saved training artifact bundle to', bundle_path)
            print(json.dumps(tensorboard_status_snapshot(), indent=2))

    launch_btn.on_click(wrap_widget_callback('training_launch', on_launch))
    poll_btn.on_click(wrap_widget_callback('training_poll_status', on_poll))
    stop_btn.on_click(wrap_widget_callback('training_stop', on_stop))
    auto_refresh_widget.observe(wrap_widget_callback('training_auto_refresh_toggle', on_auto_refresh), names='value')
    show_tb_btn.on_click(wrap_widget_callback('training_show_tensorboard', on_show_tb))
    show_dash_btn.on_click(wrap_widget_callback('training_plot_dashboard', on_show_dash))
    dump_logs_btn.on_click(wrap_widget_callback('training_dump_latest_logs', on_dump_latest_logs))
    bundle_artifacts_btn.on_click(wrap_widget_callback('training_bundle_artifacts', on_bundle_training_artifacts))

    controls = widgets.VBox([
        resume_widget,
        inline_tb_widget,
        dashboard_widget,
        widgets.HBox([auto_refresh_widget, refresh_seconds_widget]),
        summary_html,
        tensorboard_html,
        widgets.HBox([launch_btn, poll_btn, stop_btn, dump_logs_btn]),
        widgets.HBox([show_tb_btn, show_dash_btn, bundle_artifacts_btn]),
        output,
    ])
    display(controls)
    render_summary_card()
    render_tensorboard_status()


## Resume + preflight-aware tuning helpers
Hydra already writes `preflight_cache.json`, `latest_state.yaml`, and `latest_model.mpk`. These helpers let the notebook inspect that state, wire resume automatically, and optionally apply preflight-selected runtime values back into the YAML.

In [ ]:
def bc_artifact_dir() -> pathlib.Path:
    return OUTPUT_DIR / 'bc'

def preflight_cache_path() -> pathlib.Path:
    return bc_artifact_dir() / 'preflight_cache.json'

def latest_resume_state_path() -> pathlib.Path:
    return bc_artifact_dir() / 'latest_state.yaml'

def latest_model_checkpoint_path() -> pathlib.Path:
    return bc_artifact_dir() / 'latest_model.mpk'

def load_preflight_cache() -> dict | None:
    path = preflight_cache_path()
    if not path.exists():
        return None
    return json.loads(path.read_text())

def load_resume_state() -> dict | None:
    maybe_install_py_package('yaml')
    import yaml

    path = latest_resume_state_path()
    if not path.exists():
        return None
    return yaml.safe_load(path.read_text())

def load_current_config_text() -> str:
    return CONFIG_PATH.read_text()

def set_or_insert_yaml_key(text: str, key: str, value: str) -> str:
    lines = text.splitlines()
    replaced = False
    for idx, line in enumerate(lines):
        if line.startswith(f'{key}:'):
            lines[idx] = f'{key}: {value}'
            replaced = True
            break
    if not replaced:
        lines.append(f'{key}: {value}')
    return '\n'.join(lines).rstrip() + '\n'

def apply_preflight_runtime_to_config() -> None:
    cache = load_preflight_cache()
    if cache is None:
        raise RuntimeError('preflight cache missing; run preflight first')
    global TRAIN_MICROBATCH_SIZE, VALIDATION_MICROBATCH_SIZE, NUM_THREADS, BUFFER_GAMES, BUFFER_SAMPLES, ARCHIVE_QUEUE_BOUND
    runtime = cache['runtime']
    selected = runtime['selected']
    loader = runtime['loader']
    TRAIN_MICROBATCH_SIZE = int(selected['train_microbatch_size'])
    VALIDATION_MICROBATCH_SIZE = int(selected['validation_microbatch_size'])
    NUM_THREADS = int(loader['num_threads']) if loader.get('num_threads') is not None else None
    BUFFER_GAMES = int(loader['buffer_games'])
    BUFFER_SAMPLES = int(loader['buffer_samples'])
    ARCHIVE_QUEUE_BOUND = int(loader['archive_queue_bound'])
    text = load_current_config_text()
    text = set_or_insert_yaml_key(text, 'microbatch_size', str(selected['train_microbatch_size']))
    text = set_or_insert_yaml_key(text, 'validation_microbatch_size', str(selected['validation_microbatch_size']))
    text = set_or_insert_yaml_key(text, 'buffer_games', str(loader['buffer_games']))
    text = set_or_insert_yaml_key(text, 'buffer_samples', str(loader['buffer_samples']))
    text = set_or_insert_yaml_key(text, 'archive_queue_bound', str(loader['archive_queue_bound']))
    if loader.get('num_threads') is not None:
        text = set_or_insert_yaml_key(text, 'num_threads', str(loader['num_threads']))
    CONFIG_PATH.write_text(text)
    print('applied preflight runtime to config from', preflight_cache_path())

def configure_resume_from_latest_checkpoint() -> None:
    checkpoint = latest_model_checkpoint_path()
    state = latest_resume_state_path()
    if not checkpoint.exists() or not state.exists():
        raise RuntimeError('latest checkpoint/state missing; cannot wire resume yet')
    text = load_current_config_text()
    text = set_or_insert_yaml_key(text, 'resume_checkpoint', str(checkpoint))
    CONFIG_PATH.write_text(text)
    print('configured resume_checkpoint =', checkpoint)

def clear_resume_from_config() -> None:
    lines = []
    for line in load_current_config_text().splitlines():
        if not line.startswith('resume_checkpoint:'):
            lines.append(line)
    CONFIG_PATH.write_text('\n'.join(lines).rstrip() + '\n')
    print('cleared resume_checkpoint from config')

print('preflight cache =', preflight_cache_path())
print('resume state    =', latest_resume_state_path())
print('latest model    =', latest_model_checkpoint_path())
print('loaded preflight cache?', load_preflight_cache() is not None)
print('loaded resume state?   ', load_resume_state() is not None)


## Live hardware monitor helpers
These are host-side checks. Hydra does not expose fake telemetry flags, so the notebook watches the actual Kaggle machine while training runs.

In [ ]:
@dataclass
class StatSnapshot:
    label: str
    payload: str

def safe_capture(cmd: list[str]) -> str:
    try:
        out = subprocess.run(cmd, text=True, capture_output=True, check=False)
        if out.returncode == 0:
            return out.stdout.strip() or '<no output>'
        err = out.stderr.strip()
        return f"<exit={out.returncode}> {err or out.stdout.strip() or 'no detail'}"
    except FileNotFoundError:
        return '<command not found>'

def gpu_snapshot() -> StatSnapshot:
    query = 'timestamp,name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw'
    payload = safe_capture([
        'nvidia-smi',
        f'--query-gpu={query}',
        '--format=csv,noheader,nounits',
    ])
    return StatSnapshot('GPU', payload)

def cpu_mem_snapshot() -> StatSnapshot:
    payload = safe_capture([
        'bash', '-lc',
        "top -bn1 | grep 'Cpu(s)'; free -h | sed -n '2p'",
    ])
    return StatSnapshot('Host CPU/RAM', payload)

def pretty_snapshot() -> str:
    sections = [gpu_snapshot(), cpu_mem_snapshot()]
    return '\n\n'.join(f'[{section.label}]\n{section.payload}' for section in sections)

print(pretty_snapshot())


## TensorBoard helpers
Hydra writes TensorBoard files under `output_dir/bc/tb/run_g...`. Use inline TensorBoard from inside Kaggle.

In [ ]:
TENSORBOARD_PORT = 6006
ENABLE_CLOUDFLARE_TUNNEL = False
TENSORBOARD_HOST = '127.0.0.1'

def latest_tb_session_dir() -> pathlib.Path | None:
    tb_root = OUTPUT_DIR / 'bc' / 'tb'
    if not tb_root.exists():
        return None
    runs = sorted([path for path in tb_root.iterdir() if path.is_dir()])
    return runs[-1] if runs else None

def launch_tensorboard(logdir: pathlib.Path | None = None) -> subprocess.Popen:
    logdir = logdir or latest_tb_session_dir()
    if logdir is None:
        raise RuntimeError('no TensorBoard run dir found yet under output/bc/tb')
    maybe_install_py_package('tensorboard')
    cmd = [
        sys.executable, '-m', 'tensorboard.main',
        '--logdir', str(logdir),
        '--host', TENSORBOARD_HOST,
        '--port', str(TENSORBOARD_PORT),
    ]
    print('$', ' '.join(shlex.quote(part) for part in cmd))
    return subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

def stream_process_output(proc: subprocess.Popen, label: str, seconds: float = 8.0) -> None:
    start = time.time()
    while time.time() - start < seconds:
        line = proc.stdout.readline()
        if not line:
            if proc.poll() is not None:
                break
            time.sleep(0.2)
            continue
        print(f'[{label}] {line.rstrip()}')

def show_inline_tensorboard(logdir: pathlib.Path | None = None, port: int = TENSORBOARD_PORT) -> None:
    logdir = logdir or latest_tb_session_dir()
    if logdir is None:
        raise RuntimeError('no TensorBoard run dir found yet under output/bc/tb')
    try:
        from IPython import get_ipython
    except ModuleNotFoundError as err:
        raise RuntimeError('IPython is required for inline TensorBoard display') from err
    maybe_install_py_package('tensorboard')
    ip = get_ipython()
    if ip is None:
        raise RuntimeError('inline TensorBoard only works inside a live notebook kernel')
    ip.run_line_magic('load_ext', 'tensorboard')
    ip.run_line_magic('tensorboard', f'--logdir {logdir} --port {port}')

def tensorboard_status_snapshot() -> dict[str, Any]:
    logdir = latest_tb_session_dir()
    event_files = [] if logdir is None else sorted(path.name for path in logdir.rglob('events.out.tfevents.*'))
    try:
        __import__('tensorboard')
        tensorboard_available = True
        tensorboard_error = None
    except ModuleNotFoundError as err:
        tensorboard_available = False
        tensorboard_error = repr(err)
    return {
        'logdir': str(logdir) if logdir is not None else None,
        'exists': logdir is not None and logdir.exists(),
        'event_file_count': len(event_files),
        'event_files_preview': event_files[:10],
        'inline_available': tensorboard_available,
        'tensorboard_error': tensorboard_error,
    }

def write_training_artifact_bundle() -> pathlib.Path:
    ensure_persistent_state_dirs()
    candidates = [
        OUTPUT_DIR / 'bc' / 'training_log.jsonl',
        OUTPUT_DIR / 'bc' / 'step_log.jsonl',
        preflight_cache_path(),
        latest_resume_state_path(),
        latest_model_checkpoint_path(),
    ]
    tb_dir = latest_tb_session_dir()
    if tb_dir is not None and tb_dir.exists():
        candidates.extend(path for path in tb_dir.rglob('*') if path.is_file())
    with zipfile.ZipFile(TRAINING_ARTIFACT_BUNDLE_PATH, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for path in candidates:
            if not path.exists() or not path.is_file():
                continue
            if path.is_relative_to(REPO_ROOT):
                arcname = f"repo/{path.relative_to(REPO_ROOT).as_posix()}"
            else:
                arcname = f"misc/{path.name}"
            zf.write(path, arcname=arcname)
    return TRAINING_ARTIFACT_BUNDLE_PATH

print('latest TensorBoard run dir =', latest_tb_session_dir())


## Launch baseline BC training with live utilization
This cell streams Hydra training logs and also prints machine/container snapshots every few seconds.

Bottleneck reads:
- high GPU util + high VRAM use = good, GPU is being fed
- low GPU util + hot CPU / busy docker stats = data loader or host bottleneck
- low GPU util + low CPU util = config/path issue or training stalled early
- OOM / huge VRAM pressure = preflight picked too aggressive a point or your top-level batch is too spicy


In [ ]:
def enqueue_stream(stream, queue: Queue, prefix: str, log_callback: Callable[[str, str], None] | None = None):
    try:
        for line in iter(stream.readline, ''):
            if not line:
                break
            stripped = line.rstrip()
            if log_callback is not None:
                log_callback(prefix, stripped)
            queue.put((prefix, stripped))
    finally:
        stream.close()

def start_training_process() -> subprocess.Popen:
    global ACTIVE_TRAINING_PROCESS, ACTIVE_TRAINING_QUEUE, ACTIVE_TRAINING_THREADS
    if ACTIVE_TRAINING_PROCESS is not None and ACTIVE_TRAINING_PROCESS.poll() is None:
        raise RuntimeError('training process is already running')
    cmd = active_train_command()
    print('$', ' '.join(shlex.quote(part) for part in cmd))
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
    )
    queue: Queue = Queue()
    threads = [
        threading.Thread(target=enqueue_stream, args=(proc.stdout, queue, 'stdout', append_training_stream_output), daemon=True),
        threading.Thread(target=enqueue_stream, args=(proc.stderr, queue, 'stderr', append_training_stream_output), daemon=True),
    ]
    for thread in threads:
        thread.start()
    ACTIVE_TRAINING_PROCESS = proc
    ACTIVE_TRAINING_QUEUE = queue
    ACTIVE_TRAINING_THREADS = threads
    persist_process_state('training', proc, cmd, TRAINING_STREAM_LOG_PATH, TRAINING_STREAM_JSONL_PATH)
    return proc

def stop_training_process() -> None:
    proc = ACTIVE_TRAINING_PROCESS
    stop_auto_refresh('training')
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
            stop_signal = 'terminated'
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=5)
            stop_signal = 'killed'
        finalize_process_state('training', proc, stop_signal=stop_signal)
        return
    recovered = recover_process_status('training', None)
    if recovered['running']:
        stop_signal = stop_process_by_pid(recovered['pid'])
        finalize_process_state('training', None, stop_signal=stop_signal)
        return
    print('no active training process')


def drain_training_output(max_lines: int = 200) -> list[str]:
    lines = []
    queue = ACTIVE_TRAINING_QUEUE
    if queue is None:
        return lines
    while len(lines) < max_lines:
        try:
            prefix, line = queue.get_nowait()
            lines.append(f'[{prefix}] {line}')
        except Empty:
            break
    return lines

def start_preflight_process() -> subprocess.Popen:
    global ACTIVE_PREFLIGHT_PROCESS, ACTIVE_PREFLIGHT_QUEUE, ACTIVE_PREFLIGHT_THREADS
    proc = ACTIVE_PREFLIGHT_PROCESS
    if proc is not None and proc.poll() is None:
        raise RuntimeError('preflight process is already running')
    cmd = active_train_command('--preflight')
    print('$', ' '.join(shlex.quote(part) for part in cmd))
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
    )
    queue: Queue = Queue()
    threads = [
        threading.Thread(target=enqueue_stream, args=(proc.stdout, queue, 'stdout', append_preflight_stream_output), daemon=True),
        threading.Thread(target=enqueue_stream, args=(proc.stderr, queue, 'stderr', append_preflight_stream_output), daemon=True),
    ]
    for thread in threads:
        thread.start()
    ACTIVE_PREFLIGHT_PROCESS = proc
    ACTIVE_PREFLIGHT_QUEUE = queue
    ACTIVE_PREFLIGHT_THREADS = threads
    persist_process_state('preflight', proc, cmd, PREFLIGHT_STREAM_LOG_PATH, PREFLIGHT_STREAM_JSONL_PATH)
    return proc

def stop_preflight_process() -> None:
    proc = ACTIVE_PREFLIGHT_PROCESS
    stop_auto_refresh('preflight')
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
            stop_signal = 'terminated'
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=5)
            stop_signal = 'killed'
        finalize_process_state('preflight', proc, stop_signal=stop_signal)
        return
    recovered = recover_process_status('preflight', None)
    if recovered['running']:
        stop_signal = stop_process_by_pid(recovered['pid'])
        finalize_process_state('preflight', None, stop_signal=stop_signal)
        return
    print('no active preflight process')

def drain_preflight_output(max_lines: int = 200) -> list[str]:
    lines = []
    queue = ACTIVE_PREFLIGHT_QUEUE
    if queue is None:
        return lines
    while len(lines) < max_lines:
        try:
            prefix, line = queue.get_nowait()
            lines.append(f'[{prefix}] {line}')
        except Empty:
            break
    return lines

def preflight_status_snapshot() -> dict[str, Any]:
    recovered = recover_process_status('preflight', ACTIVE_PREFLIGHT_PROCESS)
    cache = load_preflight_cache()
    return {
        'running': recovered['running'],
        'pid': recovered['pid'],
        'returncode': recovered['returncode'],
        'status_source': recovered['source'],
        'process_state': recovered['state'],
        'preflight_cache_exists': preflight_cache_path().exists(),
        'preflight_cache_loaded': cache is not None,
        'selected_runtime_present': bool((cache or {}).get('runtime', {}).get('selected')),
    }

def training_status_snapshot() -> dict[str, Any]:
    recovered = recover_process_status('training', ACTIVE_TRAINING_PROCESS)
    return {
        'running': recovered['running'],
        'pid': recovered['pid'],
        'returncode': recovered['returncode'],
        'status_source': recovered['source'],
        'process_state': recovered['state'],
        'preflight_cache_exists': preflight_cache_path().exists(),
        'resume_state_exists': latest_resume_state_path().exists(),
        'latest_checkpoint_exists': latest_model_checkpoint_path().exists(),
    }

def run_training_with_monitoring() -> int:
    try:
        proc = start_training_process()
        queue = ACTIVE_TRAINING_QUEUE
        threads = ACTIVE_TRAINING_THREADS

        last_snapshot = 0.0
        while proc.poll() is None:
            now = time.time()
            if now - last_snapshot >= MONITOR_INTERVAL_SEC:
                print('\n=== hardware snapshot ===')
                print(pretty_snapshot())
                print('=== end hardware snapshot ===\n')
                last_snapshot = now
            try:
                while True:
                    prefix, line = queue.get_nowait()
                    print(f'[{prefix}] {line}')
            except Empty:
                pass
            time.sleep(0.2)

        while True:
            try:
                prefix, line = queue.get_nowait()
                print(f'[{prefix}] {line}')
            except Empty:
                break
        for thread in threads:
            thread.join(timeout=1)
        print('training exit code =', proc.returncode)
        return proc.returncode
    finally:
        pass

# Uncomment when you are ready to burn the GPU.
# exit_code = run_training_with_monitoring()
# if exit_code != 0:
#     raise RuntimeError(f'training failed with exit code {exit_code}')


In [ ]:
def maybe_install_plotting_stack() -> None:
    for package in ['matplotlib', 'pandas']:
        maybe_install_py_package(package)

def load_bc_logs():
    maybe_install_plotting_stack()
    import pandas as pd

    step_rows = read_json_lines(OUTPUT_DIR / 'bc' / 'step_log.jsonl')
    training_rows = read_json_lines(OUTPUT_DIR / 'bc' / 'training_log.jsonl')
    step_df = pd.DataFrame(step_rows)
    training_df = pd.DataFrame(training_rows)
    return step_df, training_df

def _numeric_series(df, column):
    if column not in df.columns:
        return None
    series = df[column]
    if hasattr(series, 'dtype') and str(series.dtype) == 'object':
        try:
            series = series.astype(float)
        except Exception:
            return None
    return series

def plot_bc_dashboard():
    maybe_install_plotting_stack()
    import matplotlib.pyplot as plt

    step_df, training_df = load_bc_logs()
    if step_df.empty and training_df.empty:
        raise RuntimeError('no BC JSONL logs found yet under output/bc')

    x_step = step_df['global_step'] if 'global_step' in step_df.columns else None
    x_epoch = training_df['epoch'] if 'epoch' in training_df.columns else None

    fig, axes = plt.subplots(3, 2, figsize=(16, 14))
    axes = axes.flatten()

    chart_specs = [
        ('Core losses', [
            ('train_total_loss', step_df, x_step),
            ('val_total_loss', step_df, x_step),
            ('val_policy_loss', step_df, x_step),
            ('best_val_policy_loss', step_df, x_step),
        ]),
        ('Agreement + LR', [
            ('train_policy_agreement', step_df, x_step),
            ('val_policy_agreement', step_df, x_step),
            ('best_val_agreement', step_df, x_step),
            ('lr', step_df, x_step),
        ]),
        ('Train loss breakdown A', [
            ('train_loss_policy', step_df, x_step),
            ('train_loss_value', step_df, x_step),
            ('train_loss_grp', step_df, x_step),
            ('train_loss_tenpai', step_df, x_step),
        ]),
        ('Train loss breakdown B', [
            ('train_loss_danger', step_df, x_step),
            ('train_loss_opp_next', step_df, x_step),
            ('train_loss_score_pdf', step_df, x_step),
            ('train_loss_score_cdf', step_df, x_step),
        ]),
        ('Epoch summaries', [
            ('train_total_loss', training_df, x_epoch),
            ('val_total_loss', training_df, x_epoch),
            ('val_policy_agreement', training_df, x_epoch),
            ('num_batches', training_df, x_epoch),
        ]),
        ('Samples + batches', [
            ('global_step', step_df, x_step),
            ('epoch', step_df, x_step),
        ]),
    ]

    for axis, (title, series_specs) in zip(axes, chart_specs):
        plotted = False
        for column, df, x_values in series_specs:
            if df.empty or x_values is None:
                continue
            series = _numeric_series(df, column)
            if series is None:
                continue
            values = [value for value in series.tolist()]
            if not values or all((value is None) or (isinstance(value, float) and math.isnan(value)) for value in values):
                continue
            axis.plot(x_values, series, label=column)
            plotted = True
        axis.set_title(title)
        axis.grid(True, alpha=0.3)
        if plotted:
            axis.legend()
        else:
            axis.text(0.5, 0.5, 'no data yet', ha='center', va='center', transform=axis.transAxes)

    plt.tight_layout()
    plt.show()


## Main lane
Just run all cells once, then use these in order:

1. `Notebook UI: preflight tuning`
2. `Notebook UI: training launch`

Everything below is helper code for those UIs, not another path.

In [ ]:
build_preflight_ui()
build_training_ui()
